# NB_P25_Backfill_Ingest -- Project 25 (Meridian Pay), Session 3

**What this notebook does, in order:**
1. Writes the generator package (`01_Source/generator/`) to local disk from
   embedded source, then imports it -- identical logic to `backfill.py`,
   proven reproducible twice already (Session 2 and Session 3, both matched
   `Landing_Manifest.csv` md5 `cf1c270b8440bde4c779dd127550e168` exactly).
2. Generates the full backfill in memory (no local parquet write -- this
   notebook ingests straight from the DataFrames the generator produces).
3. Ingests every stream into its target Eventhouse table via the Kusto Python
   SDK, using this notebook's own AAD context (`notebookutils.credentials`) --
   no service principal or connection string needs to be pasted in.
4. Prints `Landing_Manifest.csv` and writes it into a `landing_manifest` KQL
   table for durable, queryable provenance (no Lakehouse exists yet in this
   project as of Session 3 -- see the Session 3 handover note on this).

**Before running:** run `02_KQL/00_raw_and_staging_tables.kql` through
`02_KQL/04_functions.kql` in the KQL Queryset first (`KustoQueryWorkbench_1`)
-- this notebook only INGESTS, it does not create tables or attach update
policies. Update policies only fire on ingestion that happens AFTER they are
attached, so table/policy creation must come first.

**Fill in the two values in the Parameters cell below** (`CLUSTER_URI`,
`DATABASE`) from Workspace -> `EH_MeridianPay` -> Settings -> "Copy query URI"
before running. Everything else has a working default.

**Smoke-test first.** Set `ROW_LIMIT` to a small number (e.g. `5000`) for the
first run to prove the whole pipeline end-to-end cheaply, confirm counts in
the KQL Queryset, THEN re-run with `ROW_LIMIT = None` for the full 9.7m-event
backfill. Re-running is safe -- every raw table's natural key (`auth_id`,
`telemetry_id`, `dispute_id`) is deduplicated by the rule-1 update policy
logic downstream in `auth_curated`/`telemetry_curated`/`dispute_curated`, so a
second full ingest just gets deduplicated away in the curated layer (though it
does inflate the RAW row counts -- if you smoke-test, consider dropping and
re-creating the raw tables before the full run, or accept that raw counts will
run slightly ahead of `Landing_Manifest.csv` and note that when reconciling).


In [ ]:
# Cell 1 -- packages not preinstalled in the default Fabric runtime.
#
# VERSIONS ARE PINNED, AND THE PIN ON azure-identity IS THE IMPORTANT ONE.
# Installing azure-kusto-data unpinned pulls the latest azure-identity, which
# needs azure-core >= 1.31 for AccessTokenInfo. Fabric's trident_env ships an
# older azure-core, so the import chain dies at Cell 5 with:
#   ImportError: cannot import name 'AccessTokenInfo' from 'azure.core.credentials'
# Pinning azure-identity below that boundary means pip never needs to touch
# azure-core -- which matters, because upgrading azure-core in a Fabric runtime
# risks breaking notebookutils and anything else built against it. Fixing the
# leaf dependency is safer than upgrading the shared one.
#
# This combination was verified against azure-core 1.30.2 before shipping.
#
# subprocess, not %pip: magics are not valid standalone Python and fail this
# project's build_notebook.py ast.parse gate, which exists to catch errors here
# rather than at Fabric import time.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "azure-identity==1.17.1",
                "azure-kusto-data==4.6.1",
                "azure-kusto-ingest==4.6.1"], check=True)

import importlib.metadata as _md
for _p in ("azure-core", "azure-identity", "azure-kusto-data", "azure-kusto-ingest"):
    try:
        print(f"  {_p:22s} {_md.version(_p)}")
    except Exception as _e:
        print(f"  {_p:22s} NOT FOUND ({_e})")
print()
print("If you have ALREADY run this notebook once and hit the AccessTokenInfo")
print("ImportError, RESTART THE SESSION before continuing -- the broken")
print("azure.identity is cached in sys.modules and a reinstall alone will not")
print("dislodge it. Fabric: the '...' menu on the notebook toolbar ->")
print("'Restart session', then Run all from the top.")


In [ ]:
# Cell 2 -- PARAMETERS.
#
# Kusto uses TWO endpoints and they are not interchangeable:
#   CLUSTER_URI = the "Query URI"     -> running queries and control commands
#   INGEST_URI  = the "Ingestion URI" -> QueuedIngestClient posts data here
# Fabric shows both on the Eventhouse item's System overview page, one above
# the other. Copy both. If you leave INGEST_URI blank, the cell below derives
# it by prefixing "ingest-" to the cluster host, which is the standard mapping
# -- but paste the real one if Fabric shows something different, because a
# wrong ingest endpoint fails in a confusing way (auth succeeds, ingestion
# silently never lands).
CLUSTER_URI = "https://REPLACE-WITH-QUERY-URI.kusto.fabric.microsoft.com"
INGEST_URI = ""    # optional; blank = derive from CLUSTER_URI

DATABASE = "EH_MeridianPay"
RUN_ID = "backfill-v1"
MASTER_SEED = 250817
ROW_LIMIT = 5000   # None for the full backfill; an int for a cheap smoke test first
BATCH_ROWS = 250000   # ingestion batch size for the two large streams


In [ ]:
# Cell 3 -- write the generator package to local disk (base64-embedded
# source, byte-identical to 01_Source/generator/ in the repo -- avoids any
# quote-escaping risk between the generator's own source and this cell's
# string literals).
import base64, os
GEN_DIR = "/tmp/p25_generator/generator"
os.makedirs(GEN_DIR, exist_ok=True)
_FILES = {}
_b64___init___py = (
    "IiIiUHJvamVjdCAyNSAoTWVyaWRpYW4gUGF5KSBzeW50aGV0aWMgZGF0YSBnZW5lcmF0b3IuCgpTaW5nbGUgbW9kdWxlIHNoYXJlZCBieSB0aGUgbm90ZWJvb2sgYmFja2ZpbGwgcm91dGUgKGJhY2tmaWxsLnB5KSBhbmQgdGhlIGxvY2FsClB5dGhvbiBsaXZlLXJlcGxheSByb3V0ZSAobGl2ZV9yZXBsYXkucHkpLiBTZWUgLi4vR0VORVJBVE9SX1NQRUMubWQgZm9yIHRoZSBmdWxsCmRlc2lnbi4gTm90aGluZyBpbiB0aGlzIHBhY2thZ2UgbWF5IGNhbGwgZGF0ZXRpbWUubm93KCksIHRpbWUudGltZSgpIG9yIHRoZSBzdGRsaWIKYHJhbmRvbWAgbW9kdWxlIGZvciBhbnl0aGluZyB0aGF0IGVuZHMgdXAgaW4gYW4gb3V0cHV0IHJvdyAtLSBzZWUgY29yZS5weS4KIiIiCgpmcm9tIC5jb3JlIGltcG9ydCBHRU5FUkFUT1JfVkVSU0lPTiwgTUFTVEVSX1NFRUQsIFNJTV9OT1cKCl9fYWxsX18gPSBbIkdFTkVSQVRPUl9WRVJTSU9OIiwgIk1BU1RFUl9TRUVEIiwgIlNJTV9OT1ciXQo="
)
_FILES["__init__.py"] = _b64___init___py
_b64_core_py = (
    "IiIiU2VlZCwgdmVyc2lvbiBhbmQgUk5HLWZhY3RvcnkgY29uc3RhbnRzLiBTZWUgR0VORVJBVE9SX1NQRUMubWQgc2VjdGlvbiAxLgoKSGFyZCBydWxlIGVuZm9yY2VkIGJ5IGNvbnZlbnRpb24gYWNyb3NzIHRoaXMgd2hvbGUgcGFja2FnZTogbm8gZnVuY3Rpb24gdGhhdApjb250cmlidXRlcyB0byBhbiBvdXRwdXQgcm93IG1heSBjYWxsIGRhdGV0aW1lLm5vdygpLCB0aW1lLnRpbWUoKSBvciB0aGUgc3RkbGliCmByYW5kb21gIG1vZHVsZS4gVGhlIHNpbXVsYXRpb24gY2xvY2sgKFNJTV9OT1cpIGlzIGEgZml4ZWQgY29uc3RhbnQuIFdhbGwtY2xvY2sKcHJvdmVuYW5jZSBiZWxvbmdzIG9ubHkgaW4gdGhlIF9ydW5fbWV0YS5qc29uIHNpZGVjYXIsIHdyaXR0ZW4gb3V0c2lkZSB0aGUKcmVwcm9kdWNpYmlsaXR5LWRpZmZlZCBvdXRwdXQgdHJlZS4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgaGFzaGxpYgppbXBvcnQgZGF0ZXRpbWUgYXMgX2R0CgppbXBvcnQgbnVtcHkgYXMgbnAKCkdFTkVSQVRPUl9WRVJTSU9OID0gIjEuMC4wIgpNQVNURVJfU0VFRCA9IDI1MDgxNyAgIyBkYXRlIHRoaXMgc3BlYyB3YXMgZnJvemVuIChZWU1NREQpLCBkb2N1bWVudGVkIG5vdCBhcmJpdHJhcnkKCiMgRW50aXJlbHkgc3ludGhldGljIHNpbXVsYXRpb24gIm5vdyIuIFRoZSA5MC1kYXkgYXV0aCBiYWNrZmlsbCB3aW5kb3cgYW5kIHRoZQojIDMwLWRheSB0ZWxlbWV0cnkgYmFja2ZpbGwgd2luZG93IGFyZSBib3RoIGRlZmluZWQgcmVsYXRpdmUgdG8gdGhpcyBjb25zdGFudC4KIyBBbGwgdGltZXN0YW1wcyBpbiB0aGlzIHBhY2thZ2UgYXJlIFVUQyBieSBjb252ZW50aW9uIGFuZCBzdG9yZWQgdHotbmFpdmUKIyAoZG9jdW1lbnRlZCBvbmNlLCBoZXJlLCByYXRoZXIgdGhhbiBjYXJyeWluZyBhIHR6IG9iamVjdCB0aHJvdWdoIGV2ZXJ5CiMgdmVjdG9yaXplZCBudW1weS9wYW5kYXMgb3BlcmF0aW9uKS4KU0lNX05PVyA9IF9kdC5kYXRldGltZSgyMDI2LCA4LCAxNywgMCwgMCwgMCkKCkFVVEhfQkFDS0ZJTExfREFZUyA9IDkwClRFTEVNRVRSWV9CQUNLRklMTF9EQVlTID0gMzAKCgpkZWYgc3RyZWFtX3NlZWQoc3RyZWFtX25hbWU6IHN0ciwgbWFzdGVyX3NlZWQ6IGludCA9IE1BU1RFUl9TRUVEKSAtPiBpbnQ6CiAgICAiIiJTdGFibGUsIGRldGVybWluaXN0aWMgcGVyLXN0cmVhbSBzZWVkIGRlcml2ZWQgZnJvbSAobWFzdGVyX3NlZWQsIHN0cmVhbV9uYW1lKS4KCiAgICBVc2luZyBhIGhhc2ggcmF0aGVyIHRoYW4gZS5nLiBgbWFzdGVyX3NlZWQgKyBsZW4oc3RyZWFtX25hbWUpYCBtZWFucyBldmVyeQogICAgc3RyZWFtIGlzIGluZGVwZW5kZW50bHkgcmVwcm9kdWNpYmxlIHJlZ2FyZGxlc3Mgb2Ygd2hhdCBvcmRlciBjYWxsZXJzIGFzawogICAgZm9yIFJOR3MgaW4gLS0gaW1wb3J0YW50IG9uY2UgYmFja2ZpbGwucHkgYW5kIGxpdmVfcmVwbGF5LnB5IHJ1biBhcyBzZXBhcmF0ZQogICAgcHJvY2Vzc2VzIHRoYXQgbWF5IG5vdCB0b3VjaCBzdHJlYW1zIGluIHRoZSBzYW1lIHNlcXVlbmNlLgogICAgIiIiCiAgICBkaWdlc3QgPSBoYXNobGliLnNoYTI1NihmInttYXN0ZXJfc2VlZH06e3N0cmVhbV9uYW1lfSIuZW5jb2RlKCJhc2NpaSIpKS5kaWdlc3QoKQogICAgIyBudW1weSBHZW5lcmF0b3Igc2VlZHMgd2FudCBhIG5vbi1uZWdhdGl2ZSBpbnQgPCAyKiozMiBmb3IgcmVhZGFiaWxpdHkgaW4gbG9nczsKICAgICMgdGFrZSB0aGUgbG93IDMyIGJpdHMgb2YgdGhlIGRpZ2VzdC4KICAgIHJldHVybiBpbnQuZnJvbV9ieXRlcyhkaWdlc3RbOjRdLCAiYmlnIikKCgpkZWYgcm5nX2ZvcihzdHJlYW1fbmFtZTogc3RyLCBtYXN0ZXJfc2VlZDogaW50ID0gTUFTVEVSX1NFRUQpIC0+IG5wLnJhbmRvbS5HZW5lcmF0b3I6CiAgICAiIiJPbmUgaW5kZXBlbmRlbnQgUENHNjQgR2VuZXJhdG9yIHBlciBsb2dpY2FsIHN0cmVhbS4gTmV2ZXIgc2hhcmUgYSBnbG9iYWwgUk5HLiIiIgogICAgcmV0dXJuIG5wLnJhbmRvbS5kZWZhdWx0X3JuZyhzdHJlYW1fc2VlZChzdHJlYW1fbmFtZSwgbWFzdGVyX3NlZWQpKQoKCmRlZiB0ZXJtaW5hbF9zZWVkKHRlcm1pbmFsX2lkOiBzdHIsIG1hc3Rlcl9zZWVkOiBpbnQgPSBNQVNURVJfU0VFRCkgLT4gaW50OgogICAgIiIiUGVyLXRlcm1pbmFsIGRldGVybWluaXN0aWMgc2VlZCwgdXNlZCBmb3IgZS5nLiB0aGUgZml4ZWQgY2xvY2stc2tldyBvZmZzZXQKICAgIChEUSBydWxlIDQpIHNvIHRoZSBzYW1lIHRlcm1pbmFsIGdldHMgdGhlIHNhbWUgb2Zmc2V0IGluIGV2ZXJ5IHJ1biBhbmQgaW4KICAgIGJvdGggdGhlIGJhY2tmaWxsIGFuZCBsaXZlLXJlcGxheSByb3V0ZXMuIiIiCiAgICByZXR1cm4gc3RyZWFtX3NlZWQoZiJ0ZXJtaW5hbDp7dGVybWluYWxfaWR9IiwgbWFzdGVyX3NlZWQpCg=="
)
_FILES["core.py"] = _b64_core_py
_b64_estate_py = (
    "IiIiZGltX2VzdGF0ZSBoaWVyYXJjaHk6IE1lcmNoYW50IC0+IFN0b3JlIC0+IFRlcm1pbmFsLCBwbHVzIGRpbV9pc3N1ZXIuCgpTZWUgR0VORVJBVE9SX1NQRUMubWQgc2VjdGlvbiAyLCBpbmNsdWRpbmcgdGhlIG5vdGUgdGhhdCBJc3N1ZXIgaXMgcmVzb2x2ZWQgcGVyCnRyYW5zYWN0aW9uIHJhdGhlciB0aGFuIHN0cnVjdHVyYWxseSBuZXN0ZWQgdW5kZXIgVGVybWluYWwuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHBhbmRhcyBhcyBwZAppbXBvcnQgbnVtcHkgYXMgbnAKCmZyb20gLmNvcmUgaW1wb3J0IHJuZ19mb3IKCk5fTUVSQ0hBTlRTID0gNjAwClRBUkdFVF9TVE9SRVMgPSA5MDAKU1RPUkVfVE9MRVJBTkNFID0gMC4wMyAgIyArLy0zJSwgZGVjbGFyZWQgaW4gdGhlIHNwZWMgcmF0aGVyIHRoYW4gc2lsZW50bHkgcm91bmRlZApUQVJHRVRfVEVSTUlOQUxTID0gMTUwMApOX0lTU1VFUlMgPSAyNAoKTUNDX1BPT0wgPSBbCiAgICAoIjU0MTEiLCAiR3JvY2VyeSBTdG9yZXMiKSwKICAgICgiNTgxMiIsICJSZXN0YXVyYW50cyIpLAogICAgKCI1NTQxIiwgIkZ1ZWwgU3RhdGlvbnMiKSwKICAgICgiNTY5MSIsICJBcHBhcmVsIiksCiAgICAoIjU5OTkiLCAiU3BlY2lhbHR5IFJldGFpbCIpLAogICAgKCI1OTEyIiwgIlBoYXJtYWNpZXMiKSwKICAgICgiNTgxNCIsICJGYXN0IEZvb2QiKSwKICAgICgiNTMxMSIsICJEZXBhcnRtZW50IFN0b3JlcyIpLAogICAgKCI1NzMyIiwgIkVsZWN0cm9uaWNzIiksCiAgICAoIjUyNjEiLCAiR2FyZGVuIFN1cHBseSIpLAogICAgKCI1NjUxIiwgIkZhbWlseSBDbG90aGluZyIpLAogICAgKCI1OTQyIiwgIkJvb2sgU3RvcmVzIiksCl0KClRJRVJfV0VJR0hUUyA9IHsic21hbGwiOiAwLjYwLCAibWVkaXVtIjogMC4zMCwgImxhcmdlIjogMC4xMH0KCgpkZWYgX21lcmNoYW50X3RpZXJzKHJuZzogbnAucmFuZG9tLkdlbmVyYXRvcikgLT4gbnAubmRhcnJheToKICAgIHRpZXJzID0gbGlzdChUSUVSX1dFSUdIVFMua2V5cygpKQogICAgcHJvYnMgPSBsaXN0KFRJRVJfV0VJR0hUUy52YWx1ZXMoKSkKICAgIHJldHVybiBybmcuY2hvaWNlKHRpZXJzLCBzaXplPU5fTUVSQ0hBTlRTLCBwPXByb2JzKQoKCmRlZiBfc3RvcmVfY291bnRzX3Blcl9tZXJjaGFudCh0aWVyczogbnAubmRhcnJheSwgcm5nOiBucC5yYW5kb20uR2VuZXJhdG9yKSAtPiBucC5uZGFycmF5OgogICAgY291bnRzID0gbnAuZW1wdHkoTl9NRVJDSEFOVFMsIGR0eXBlPWludCkKICAgIGZvciBpLCB0aWVyIGluIGVudW1lcmF0ZSh0aWVycyk6CiAgICAgICAgaWYgdGllciA9PSAic21hbGwiOgogICAgICAgICAgICBjb3VudHNbaV0gPSAxCiAgICAgICAgZWxpZiB0aWVyID09ICJtZWRpdW0iOgogICAgICAgICAgICBjb3VudHNbaV0gPSBpbnQocm5nLnBvaXNzb24oMikpICsgMQogICAgICAgIGVsc2U6ICAjIGxhcmdlCiAgICAgICAgICAgIGNvdW50c1tpXSA9IGludChybmcucG9pc3Nvbig0KSkgKyAyCiAgICByZXR1cm4gY291bnRzCgoKZGVmIF9maXRfc3RvcmVfdG90YWwoY291bnRzOiBucC5uZGFycmF5LCBybmc6IG5wLnJhbmRvbS5HZW5lcmF0b3IpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJEZXRlcm1pbmlzdGljYWxseSBudWRnZSBzdG9yZSBjb3VudHMgc28gdGhlIHRvdGFsIGxhbmRzIHdpdGhpbgogICAgVEFSR0VUX1NUT1JFUyArLy0gU1RPUkVfVE9MRVJBTkNFLiBBZGp1c3RtZW50cyBhbHdheXMgdG91Y2ggdGhlIG1lcmNoYW50cwogICAgd2l0aCB0aGUgbGFyZ2VzdCBjb3VudHMgZmlyc3QsIGluIGEgc3RhYmxlIChpbmRleCkgb3JkZXIsIHNvIHRoZSByZXN1bHQgaXMKICAgIHJlcHJvZHVjaWJsZSBmb3IgYSBnaXZlbiBzZWVkLiIiIgogICAgbG8gPSBpbnQoVEFSR0VUX1NUT1JFUyAqICgxIC0gU1RPUkVfVE9MRVJBTkNFKSkKICAgIGhpID0gaW50KFRBUkdFVF9TVE9SRVMgKiAoMSArIFNUT1JFX1RPTEVSQU5DRSkpCiAgICBjb3VudHMgPSBjb3VudHMuY29weSgpCiAgICBvcmRlciA9IG5wLmFyZ3NvcnQoLWNvdW50cywga2luZD0ic3RhYmxlIikgICMgbGFyZ2VzdCBmaXJzdCwgc3RhYmxlIG9yZGVyCiAgICBwb3MgPSAwCiAgICB3aGlsZSBjb3VudHMuc3VtKCkgPiBoaToKICAgICAgICBpZHggPSBvcmRlcltwb3MgJSBsZW4ob3JkZXIpXQogICAgICAgIGlmIGNvdW50c1tpZHhdID4gMToKICAgICAgICAgICAgY291bnRzW2lkeF0gLT0gMQogICAgICAgIHBvcyArPSAxCiAgICAgICAgaWYgcG9zID4gMTAgKiBsZW4ob3JkZXIpOgogICAgICAgICAgICBicmVhawogICAgcG9zID0gMAogICAgd2hpbGUgY291bnRzLnN1bSgpIDwgbG86CiAgICAgICAgaWR4ID0gb3JkZXJbcG9zICUgbGVuKG9yZGVyKV0KICAgICAgICBjb3VudHNbaWR4XSArPSAxCiAgICAgICAgcG9zICs9IDEKICAgICAgICBpZiBwb3MgPiAxMCAqIGxlbihvcmRlcik6CiAgICAgICAgICAgIGJyZWFrCiAgICByZXR1cm4gY291bnRzCgoKZGVmIF90ZXJtaW5hbF9jb3VudHNfcGVyX3N0b3JlKG5fc3RvcmVzOiBpbnQsIHJuZzogbnAucmFuZG9tLkdlbmVyYXRvcikgLT4gbnAubmRhcnJheToKICAgIGNvdW50cyA9IHJuZy5wb2lzc29uKDEuMiwgc2l6ZT1uX3N0b3JlcykgKyAxCiAgICBkaWZmID0gVEFSR0VUX1RFUk1JTkFMUyAtIGludChjb3VudHMuc3VtKCkpCiAgICBpZiBkaWZmID4gMDoKICAgICAgICAjIHBhZCB0aGUgZmlyc3QgYGRpZmZgIHN0b3JlcyAoc3RvcmVfaWQgYXNjZW5kaW5nID09IGFycmF5IGluZGV4IG9yZGVyKQogICAgICAgIGZvciBpIGluIHJhbmdlKGRpZmYpOgogICAgICAgICAgICBjb3VudHNbaSAlIG5fc3RvcmVzXSArPSAxCiAgICBlbGlmIGRpZmYgPCAwOgogICAgICAgIG5lZWQgPSAtZGlmZgogICAgICAgICMgdHJpbSBmcm9tIGxhcmdlc3Qgc3RvcmVzIGZpcnN0LCBuZXZlciBiZWxvdyAxIHRlcm1pbmFsCiAgICAgICAgb3JkZXIgPSBucC5hcmdzb3J0KC1jb3VudHMsIGtpbmQ9InN0YWJsZSIpCiAgICAgICAgaSA9IDAKICAgICAg"
    "ICB3aGlsZSBuZWVkID4gMDoKICAgICAgICAgICAgaWR4ID0gb3JkZXJbaSAlIGxlbihvcmRlcildCiAgICAgICAgICAgIGlmIGNvdW50c1tpZHhdID4gMToKICAgICAgICAgICAgICAgIGNvdW50c1tpZHhdIC09IDEKICAgICAgICAgICAgICAgIG5lZWQgLT0gMQogICAgICAgICAgICBpICs9IDEKICAgICAgICAgICAgaWYgaSA+IDIwICogbGVuKG9yZGVyKToKICAgICAgICAgICAgICAgIGJyZWFrCiAgICByZXR1cm4gY291bnRzCgoKZGVmIGJ1aWxkX2RpbV9lc3RhdGUobWFzdGVyX3NlZWQ6IGludCB8IE5vbmUgPSBOb25lKToKICAgICIiIlJldHVybnMgKGRpbV9tZXJjaGFudCwgZGltX3N0b3JlLCBkaW1fdGVybWluYWwsIGRpbV9pc3N1ZXIpIERhdGFGcmFtZXMsCiAgICBlYWNoIHRlcm1pbmFsIGNhcnJ5aW5nIGl0cyByZXNvbHZlZCBtZXJjaGFudF9pZC9zdG9yZV9pZCBkZW5vcm1hbGlzZWQuIiIiCiAgICBybmcgPSBybmdfZm9yKCJlc3RhdGUiLCBtYXN0ZXJfc2VlZCkgaWYgbWFzdGVyX3NlZWQgaXMgbm90IE5vbmUgZWxzZSBybmdfZm9yKCJlc3RhdGUiKQoKICAgIHRpZXJzID0gX21lcmNoYW50X3RpZXJzKHJuZykKICAgIG1jY19pZHggPSBybmcuaW50ZWdlcnMoMCwgbGVuKE1DQ19QT09MKSwgc2l6ZT1OX01FUkNIQU5UUykKICAgIG1lcmNoYW50X2lkcyA9IFtmIk1FUi17aSsxOjA2ZH0iIGZvciBpIGluIHJhbmdlKE5fTUVSQ0hBTlRTKV0KICAgIGRpbV9tZXJjaGFudCA9IHBkLkRhdGFGcmFtZSgKICAgICAgICB7CiAgICAgICAgICAgICJtZXJjaGFudF9pZCI6IG1lcmNoYW50X2lkcywKICAgICAgICAgICAgIm1lcmNoYW50X3RpZXIiOiB0aWVycywKICAgICAgICAgICAgIm1jYyI6IFtNQ0NfUE9PTFtqXVswXSBmb3IgaiBpbiBtY2NfaWR4XSwKICAgICAgICAgICAgIm1jY19kZXNjcmlwdGlvbiI6IFtNQ0NfUE9PTFtqXVsxXSBmb3IgaiBpbiBtY2NfaWR4XSwKICAgICAgICB9CiAgICApCgogICAgc3RvcmVfY291bnRzID0gX3N0b3JlX2NvdW50c19wZXJfbWVyY2hhbnQodGllcnMsIHJuZykKICAgIHN0b3JlX2NvdW50cyA9IF9maXRfc3RvcmVfdG90YWwoc3RvcmVfY291bnRzLCBybmcpCgogICAgc3RvcmVfcm93cyA9IFtdCiAgICBzdG9yZV9zZXEgPSAwCiAgICBmb3IgbV9pZHgsIG1faWQgaW4gZW51bWVyYXRlKG1lcmNoYW50X2lkcyk6CiAgICAgICAgZm9yIF8gaW4gcmFuZ2UoaW50KHN0b3JlX2NvdW50c1ttX2lkeF0pKToKICAgICAgICAgICAgc3RvcmVfc2VxICs9IDEKICAgICAgICAgICAgc3RvcmVfcm93cy5hcHBlbmQoCiAgICAgICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAgICAgInN0b3JlX2lkIjogZiJTVFIte3N0b3JlX3NlcTowNmR9IiwKICAgICAgICAgICAgICAgICAgICAibWVyY2hhbnRfaWQiOiBtX2lkLAogICAgICAgICAgICAgICAgfQogICAgICAgICAgICApCiAgICBkaW1fc3RvcmUgPSBwZC5EYXRhRnJhbWUoc3RvcmVfcm93cykKICAgIG5fc3RvcmVzID0gbGVuKGRpbV9zdG9yZSkKCiAgICB0ZXJtaW5hbF9jb3VudHMgPSBfdGVybWluYWxfY291bnRzX3Blcl9zdG9yZShuX3N0b3Jlcywgcm5nKQoKICAgIHRlcm1fcm93cyA9IFtdCiAgICB0ZXJtX3NlcSA9IDAKICAgIGZvciBzX2lkeCwgc3RvcmVfcm93IGluIGRpbV9zdG9yZS5pdGVycm93cygpOgogICAgICAgIGZvciBfIGluIHJhbmdlKGludCh0ZXJtaW5hbF9jb3VudHNbc19pZHhdKSk6CiAgICAgICAgICAgIHRlcm1fc2VxICs9IDEKICAgICAgICAgICAgdGVybV9yb3dzLmFwcGVuZCgKICAgICAgICAgICAgICAgIHsKICAgICAgICAgICAgICAgICAgICAidGVybWluYWxfaWQiOiBmIlRSTS17dGVybV9zZXE6MDZkfSIsCiAgICAgICAgICAgICAgICAgICAgInN0b3JlX2lkIjogc3RvcmVfcm93WyJzdG9yZV9pZCJdLAogICAgICAgICAgICAgICAgICAgICJtZXJjaGFudF9pZCI6IHN0b3JlX3Jvd1sibWVyY2hhbnRfaWQiXSwKICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgKQogICAgZGltX3Rlcm1pbmFsID0gcGQuRGF0YUZyYW1lKHRlcm1fcm93cykKCiAgICBpc3N1ZXJfaWRzID0gW2YiSVNTLXtpKzE6MDNkfSIgZm9yIGkgaW4gcmFuZ2UoTl9JU1NVRVJTKV0KICAgIGlzc3Vlcl9iaW5zID0gW2YiezQwMDAwMCArIGkgKiAxMzd9IiBmb3IgaSBpbiByYW5nZShOX0lTU1VFUlMpXSAgIyBzeW50aGV0aWMgQklOLXNoYXBlZCBzdHJpbmdzCiAgICBkaW1faXNzdWVyID0gcGQuRGF0YUZyYW1lKAogICAgICAgIHsKICAgICAgICAgICAgImlzc3Vlcl9pZCI6IGlzc3Vlcl9pZHMsCiAgICAgICAgICAgICJpc3N1ZXJfYmluIjogaXNzdWVyX2JpbnMsCiAgICAgICAgICAgICJpc3N1ZXJfbmFtZSI6IFtmIklzc3VlciBCYW5rIHtpKzE6MDJkfSIgZm9yIGkgaW4gcmFuZ2UoTl9JU1NVRVJTKV0sCiAgICAgICAgfQogICAgKQoKICAgIHJldHVybiBkaW1fbWVyY2hhbnQsIGRpbV9zdG9yZSwgZGltX3Rlcm1pbmFsLCBkaW1faXNzdWVyCg=="
)
_FILES["estate.py"] = _b64_estate_py
_b64_trading_calendar_py = (
    "IiIiUGVyLXN0b3JlIHRyYWRpbmctaG91cnMgY2FsZW5kYXIuIFNlZSBHRU5FUkFUT1JfU1BFQy5tZCBzZWN0aW9uIDMuCgpQcm9kdWNlcyBkaW1fc3RvcmVfY2FsZW5kYXIgKGZsYXQgdGFibGUsIGZvciBLUUwvUG93ZXIgQkkgam9pbnMpIHBsdXMgYSBjb21wYWN0CmFyY2hldHlwZSBtYXAgdXNlZCBpbnRlcm5hbGx5IGJ5IGF1dGhfZXZlbnRzL3RlbGVtZXRyeS9lcGlzb2RlcyBmb3IgZmFzdAp2ZWN0b3JpemVkICJpcyB0aGlzIHRpbWVzdGFtcCBpbnNpZGUgdHJhZGluZyBob3VycyIgY2hlY2tzLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBwYW5kYXMgYXMgcGQKaW1wb3J0IG51bXB5IGFzIG5wCgpmcm9tIC5jb3JlIGltcG9ydCBybmdfZm9yCgpBUkNIRVRZUEVTID0gewogICAgIyBuYW1lOiAod2VpZ2h0LCBbKG9wZW5faG91ciwgY2xvc2VfaG91ciksIC4uLl0gcGVyIG5vcm1hbCBkYXksIGNsb3NlZF9kYXlzKQogICAgInN0YW5kYXJkX3JldGFpbCI6ICgwLjU1LCBbKDgsIDIyKV0sIHNldCgpKSwKICAgICJjb252ZW5pZW5jZV8yNGgiOiAoMC4xNSwgWygwLCAyNCldLCBzZXQoKSksCiAgICAicmVzdHJpY3RlZF9mbmIiOiAoMC4yMCwgWygxMSwgMTUpLCAoMTgsIDIzKV0sIHNldCgpKSwKICAgICJ3ZWVrZGF5X29ubHkiOiAoMC4xMCwgWyg4LCAyMCldLCB7Nn0pLCAgIyBjbG9zZWQgU3VuZGF5IChkYXlfb2Zfd2Vlaz02KQp9CgpEQVlfTkFNRVMgPSBbIk1vbiIsICJUdWUiLCAiV2VkIiwgIlRodSIsICJGcmkiLCAiU2F0IiwgIlN1biJdCgoKZGVmIGFzc2lnbl9hcmNoZXR5cGVzKHN0b3JlX2lkczogbGlzdFtzdHJdLCBtYXN0ZXJfc2VlZDogaW50IHwgTm9uZSA9IE5vbmUpIC0+IHBkLlNlcmllczoKICAgIHJuZyA9IHJuZ19mb3IoImNhbGVuZGFyX2FyY2hldHlwZSIsIG1hc3Rlcl9zZWVkKSBpZiBtYXN0ZXJfc2VlZCBpcyBub3QgTm9uZSBlbHNlIHJuZ19mb3IoCiAgICAgICAgImNhbGVuZGFyX2FyY2hldHlwZSIKICAgICkKICAgIG5hbWVzID0gbGlzdChBUkNIRVRZUEVTLmtleXMoKSkKICAgIHdlaWdodHMgPSBbQVJDSEVUWVBFU1tuXVswXSBmb3IgbiBpbiBuYW1lc10KICAgIGNob2ljZSA9IHJuZy5jaG9pY2UobmFtZXMsIHNpemU9bGVuKHN0b3JlX2lkcyksIHA9d2VpZ2h0cykKICAgIHJldHVybiBwZC5TZXJpZXMoY2hvaWNlLCBpbmRleD1zdG9yZV9pZHMsIG5hbWU9ImFyY2hldHlwZSIpCgoKZGVmIGJ1aWxkX3RyYWRpbmdfaG91cnMoZGltX3N0b3JlOiBwZC5EYXRhRnJhbWUsIG1hc3Rlcl9zZWVkOiBpbnQgfCBOb25lID0gTm9uZSk6CiAgICAiIiJSZXR1cm5zIChkaW1fc3RvcmVfY2FsZW5kYXIsIGFyY2hldHlwZV9tYXApLgoKICAgIGRpbV9zdG9yZV9jYWxlbmRhcjogb25lIHJvdyBwZXIgKHN0b3JlX2lkLCBkYXlfb2Zfd2Vlaywgc2Vzc2lvbl9ubykgd2l0aAogICAgb3Blbl90aW1lL2Nsb3NlX3RpbWUgYXMgSEg6TU0gc3RyaW5ncywgb3IgaXNfY2xvc2VkPVRydWUuCiAgICBhcmNoZXR5cGVfbWFwOiBzdG9yZV9pZCAtPiBhcmNoZXR5cGUgbmFtZSwgZm9yIGZhc3QgaW50ZXJuYWwgbG9va3Vwcy4KICAgICIiIgogICAgc3RvcmVfaWRzID0gZGltX3N0b3JlWyJzdG9yZV9pZCJdLnRvbGlzdCgpCiAgICBhcmNoZXR5cGVfbWFwID0gYXNzaWduX2FyY2hldHlwZXMoc3RvcmVfaWRzLCBtYXN0ZXJfc2VlZCkKCiAgICByb3dzID0gW10KICAgIGZvciBzdG9yZV9pZCBpbiBzdG9yZV9pZHM6CiAgICAgICAgYXJjaGV0eXBlID0gYXJjaGV0eXBlX21hcFtzdG9yZV9pZF0KICAgICAgICBfLCBzZXNzaW9ucywgY2xvc2VkX2RheXMgPSBBUkNIRVRZUEVTW2FyY2hldHlwZV0KICAgICAgICBmb3IgZG93IGluIHJhbmdlKDcpOgogICAgICAgICAgICBpZiBkb3cgaW4gY2xvc2VkX2RheXM6CiAgICAgICAgICAgICAgICByb3dzLmFwcGVuZCgKICAgICAgICAgICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAgICAgICAgICJzdG9yZV9pZCI6IHN0b3JlX2lkLAogICAgICAgICAgICAgICAgICAgICAgICAiZGF5X29mX3dlZWsiOiBkb3csCiAgICAgICAgICAgICAgICAgICAgICAgICJkYXlfbmFtZSI6IERBWV9OQU1FU1tkb3ddLAogICAgICAgICAgICAgICAgICAgICAgICAic2Vzc2lvbl9ubyI6IDEsCiAgICAgICAgICAgICAgICAgICAgICAgICJpc19jbG9zZWQiOiBUcnVlLAogICAgICAgICAgICAgICAgICAgICAgICAib3Blbl90aW1lIjogTm9uZSwKICAgICAgICAgICAgICAgICAgICAgICAgImNsb3NlX3RpbWUiOiBOb25lLAogICAgICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIGZvciBzZXNzaW9uX25vLCAob3Blbl9oLCBjbG9zZV9oKSBpbiBlbnVtZXJhdGUoc2Vzc2lvbnMsIHN0YXJ0PTEpOgogICAgICAgICAgICAgICAgcm93cy5hcHBlbmQoCiAgICAgICAgICAgICAgICAgICAgewogICAgICAgICAgICAgICAgICAgICAgICAic3RvcmVfaWQiOiBzdG9yZV9pZCwKICAgICAgICAgICAgICAgICAgICAgICAgImRheV9vZl93ZWVrIjogZG93LAogICAgICAgICAgICAgICAgICAgICAgICAiZGF5X25hbWUiOiBEQVlfTkFNRVNbZG93XSwKICAgICAgICAgICAgICAgICAgICAgICAgInNlc3Npb25fbm8iOiBzZXNzaW9uX25vLAogICAgICAgICAgICAgICAgICAgICAgICAiaXNfY2xvc2VkIjogRmFsc2UsCiAgICAgICAgICAgICAgICAgICAgICAgICJvcGVuX3RpbWUiOiBmIntvcGVuX2g6MDJkfTowMCIsCiAgICAgICAgICAgICAgICAgICAgICAgICJjbG9zZV90aW1lIjogZiJ7Y2xvc2VfaDowMmR9OjAwIiBpZiBjbG9zZV9oIDwgMjQgZWxzZSAiMjQ6MDAiLAogICAgICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgICAgICkKICAgIGRpbV9zdG9yZV9jYWxlbmRhciA9IHBkLkRhdGFGcmFtZShyb3dzKQogICAgcmV0dXJuIGRpbV9zdG9yZV9jYWxlbmRhciwgYXJjaGV0eXBlX21hcAoKCmRlZiBpc190cmFkaW5nX2hvdXJzKHN0b3JlX2lkOiBz"
    "dHIsIHRpbWVzdGFtcDogInBkLlRpbWVzdGFtcCIsIGFyY2hldHlwZV9tYXA6IHBkLlNlcmllcykgLT4gYm9vbDoKICAgICIiIlNjYWxhciBoZWxwZXIgLS0gdXNlZCBieSBlcGlzb2RlIGluamVjdGlvbiB3aGVuIGl0IG5lZWRzIHRvIGJpYXMgYSBzaW5nbGUKICAgIHdpbmRvdyAoZS5nLiByZWZsZXgtMiB0ZXJtaW5hbF9jb21wcm9taXNlIHByZWZlcnJpbmcgb3V0LW9mLWhvdXJzKS4gQnVsawogICAgZ2VuZXJhdGlvbiB1c2VzIGlzX3RyYWRpbmdfaG91cnNfdmVjIGluc3RlYWQgZm9yIHNwZWVkLiIiIgogICAgYXJjaGV0eXBlID0gYXJjaGV0eXBlX21hcFtzdG9yZV9pZF0KICAgIF8sIHNlc3Npb25zLCBjbG9zZWRfZGF5cyA9IEFSQ0hFVFlQRVNbYXJjaGV0eXBlXQogICAgZG93ID0gdGltZXN0YW1wLndlZWtkYXkoKQogICAgaWYgZG93IGluIGNsb3NlZF9kYXlzOgogICAgICAgIHJldHVybiBGYWxzZQogICAgaG91ciA9IHRpbWVzdGFtcC5ob3VyICsgdGltZXN0YW1wLm1pbnV0ZSAvIDYwLjAKICAgIHJldHVybiBhbnkob3Blbl9oIDw9IGhvdXIgPCBjbG9zZV9oIGZvciBvcGVuX2gsIGNsb3NlX2ggaW4gc2Vzc2lvbnMpCgoKZGVmIGlzX3RyYWRpbmdfaG91cnNfdmVjKHN0b3JlX2lkczogbnAubmRhcnJheSwgdGltZXN0YW1wczogcGQuRGF0ZXRpbWVJbmRleCwgYXJjaGV0eXBlX21hcDogcGQuU2VyaWVzKSAtPiBucC5uZGFycmF5OgogICAgIiIiVmVjdG9yaXplZCB2ZXJzaW9uIGZvciB3aG9sZS1mcmFtZSBnZW5lcmF0aW9uLiIiIgogICAgYXJjaGV0eXBlcyA9IGFyY2hldHlwZV9tYXAucmVpbmRleChzdG9yZV9pZHMpLnRvX251bXB5KCkKICAgIGRvd3MgPSB0aW1lc3RhbXBzLndlZWtkYXkudG9fbnVtcHkoKQogICAgaG91cnMgPSB0aW1lc3RhbXBzLmhvdXIudG9fbnVtcHkoKSArIHRpbWVzdGFtcHMubWludXRlLnRvX251bXB5KCkgLyA2MC4wCgogICAgcmVzdWx0ID0gbnAuemVyb3MobGVuKHN0b3JlX2lkcyksIGR0eXBlPWJvb2wpCiAgICBmb3IgbmFtZSwgKF8sIHNlc3Npb25zLCBjbG9zZWRfZGF5cykgaW4gQVJDSEVUWVBFUy5pdGVtcygpOgogICAgICAgIG1hc2sgPSBhcmNoZXR5cGVzID09IG5hbWUKICAgICAgICBpZiBub3QgbWFzay5hbnkoKToKICAgICAgICAgICAgY29udGludWUKICAgICAgICBub3RfY2xvc2VkID0gfm5wLmlzaW4oZG93cywgbGlzdChjbG9zZWRfZGF5cykpIGlmIGNsb3NlZF9kYXlzIGVsc2UgbnAub25lcyhsZW4oc3RvcmVfaWRzKSwgZHR5cGU9Ym9vbCkKICAgICAgICBpbl9zZXNzaW9uID0gbnAuemVyb3MobGVuKHN0b3JlX2lkcyksIGR0eXBlPWJvb2wpCiAgICAgICAgZm9yIG9wZW5faCwgY2xvc2VfaCBpbiBzZXNzaW9uczoKICAgICAgICAgICAgaW5fc2Vzc2lvbiB8PSAoaG91cnMgPj0gb3Blbl9oKSAmIChob3VycyA8IGNsb3NlX2gpCiAgICAgICAgcmVzdWx0IHw9IG1hc2sgJiBub3RfY2xvc2VkICYgaW5fc2Vzc2lvbgogICAgcmV0dXJuIHJlc3VsdAo="
)
_FILES["trading_calendar.py"] = _b64_trading_calendar_py
_b64_episodes_py = (
    "IiIiSW5qZWN0ZWQgZnJhdWQvb3BlcmF0aW9uYWwgZXBpc29kZXMgLT4gZ3JvdW5kX3RydXRoLiBTZWUgR0VORVJBVE9SX1NQRUMubWQgc2VjdGlvbiA1LgoKUnVucyBCRUZPUkUgdGhlIHJhdyBzdHJlYW1zIGFyZSBnZW5lcmF0ZWQuIGF1dGhfZXZlbnRzLnB5IC8gdGVsZW1ldHJ5LnB5IHJlYWQgdGhlCnJldHVybmVkIGVwaXNvZGUgbGlzdCBhbmQgYmlhcyB0aGVpciBvdXRwdXQgaW5zaWRlIGVhY2ggd2luZG93IHNvIHRoZSBlcGlzb2RlIGlzCmFjdHVhbGx5IGRldGVjdGFibGUsIG5vdCBqdXN0IGxhYmVsbGVkLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCB1dWlkCgppbXBvcnQgcGFuZGFzIGFzIHBkCmltcG9ydCBudW1weSBhcyBucAoKZnJvbSAuY29yZSBpbXBvcnQgcm5nX2ZvciwgU0lNX05PVywgQVVUSF9CQUNLRklMTF9EQVlTLCBURUxFTUVUUllfQkFDS0ZJTExfREFZUwpmcm9tIC50cmFkaW5nX2NhbGVuZGFyIGltcG9ydCBpc190cmFkaW5nX2hvdXJzLCBBUkNIRVRZUEVTCgpOX0NBUkRfVEVTVElORyA9IDQwCk5fVEVSTUlOQUxfQ09NUFJPTUlTRSA9IDI1Ck5fSVNTVUVSX0RFR1JBREFUSU9OID0gMTIKTl9URVJNSU5BTF9EQVJLID0gNjAKCkFVVEhfV0lORE9XX1NUQVJUID0gU0lNX05PVyAtIHBkLlRpbWVkZWx0YShkYXlzPUFVVEhfQkFDS0ZJTExfREFZUykKVEVMRU1FVFJZX1dJTkRPV19TVEFSVCA9IFNJTV9OT1cgLSBwZC5UaW1lZGVsdGEoZGF5cz1URUxFTUVUUllfQkFDS0ZJTExfREFZUykKCgpkZWYgX2VwaXNvZGVfaWQocm5nOiBucC5yYW5kb20uR2VuZXJhdG9yKSAtPiBzdHI6CiAgICAjIGRldGVybWluaXN0aWMtbG9va2luZyBpZCBidWlsdCBmcm9tIHRoZSBzZWVkZWQgUk5HLCBub3QgdXVpZDQgKHdoaWNoIGlzCiAgICAjIG5vdCBzZWVkYWJsZSkgLS0gZHJhdyAxNiByYW5kb20gaGV4IG5pYmJsZXMgZnJvbSB0aGUgc2VlZGVkIGdlbmVyYXRvci4KICAgIHJldHVybiAiRVAtIiArICIiLmpvaW4oZiJ7cm5nLmludGVnZXJzKDAsIDE2KTp4fSIgZm9yIF8gaW4gcmFuZ2UoMTIpKQoKCmRlZiBfcmFuZG9tX3RzKHJuZzogbnAucmFuZG9tLkdlbmVyYXRvciwgc3RhcnQ6IHBkLlRpbWVzdGFtcCwgZW5kOiBwZC5UaW1lc3RhbXApIC0+IHBkLlRpbWVzdGFtcDoKICAgIHNwYW5fc2Vjb25kcyA9IChlbmQgLSBzdGFydCkudG90YWxfc2Vjb25kcygpCiAgICBvZmZzZXQgPSBybmcudW5pZm9ybSgwLCBzcGFuX3NlY29uZHMpCiAgICByZXR1cm4gc3RhcnQgKyBwZC5UaW1lZGVsdGEoc2Vjb25kcz1vZmZzZXQpCgoKZGVmIGluamVjdF9lcGlzb2RlcyhkaW1fdGVybWluYWw6IHBkLkRhdGFGcmFtZSwgZGltX2lzc3VlcjogcGQuRGF0YUZyYW1lLCBhcmNoZXR5cGVfbWFwOiBwZC5TZXJpZXMsIG1hc3Rlcl9zZWVkOiBpbnQgfCBOb25lID0gTm9uZSk6CiAgICAiIiJSZXR1cm5zIGEgZ3JvdW5kX3RydXRoIERhdGFGcmFtZS4gUm93IG9yZGVyIGlzIGRldGVybWluaXN0aWMgZm9yIGEgZ2l2ZW4gc2VlZC4iIiIKICAgIHJuZyA9IHJuZ19mb3IoImVwaXNvZGVzIiwgbWFzdGVyX3NlZWQpIGlmIG1hc3Rlcl9zZWVkIGlzIG5vdCBOb25lIGVsc2Ugcm5nX2ZvcigiZXBpc29kZXMiKQoKICAgIHRlcm1pbmFsX3RvX3N0b3JlID0gZGltX3Rlcm1pbmFsLnNldF9pbmRleCgidGVybWluYWxfaWQiKVsic3RvcmVfaWQiXQogICAgdGVybWluYWxfaWRzID0gZGltX3Rlcm1pbmFsWyJ0ZXJtaW5hbF9pZCJdLnRvX251bXB5KCkKICAgIGlzc3Vlcl9iaW5zID0gZGltX2lzc3VlclsiaXNzdWVyX2JpbiJdLnRvX251bXB5KCkKCiAgICByb3dzID0gW10KCiAgICAjIDEuIGNhcmRfdGVzdGluZ19idXJzdCAtLSBhbnkgdGltZSBpbiB0aGUgOTAtZGF5IGF1dGggd2luZG93CiAgICBmb3IgXyBpbiByYW5nZShOX0NBUkRfVEVTVElORyk6CiAgICAgICAgdGVybV9pZCA9IHJuZy5jaG9pY2UodGVybWluYWxfaWRzKQogICAgICAgIHN0YXJ0ID0gX3JhbmRvbV90cyhybmcsIEFVVEhfV0lORE9XX1NUQVJULCBTSU1fTk9XIC0gcGQuVGltZWRlbHRhKG1pbnV0ZXM9MjApKQogICAgICAgIGR1cmF0aW9uX21pbiA9IGludChybmcuaW50ZWdlcnMoNSwgMTYpKQogICAgICAgIGVuZCA9IHN0YXJ0ICsgcGQuVGltZWRlbHRhKG1pbnV0ZXM9ZHVyYXRpb25fbWluKQogICAgICAgIHJvd19jb3VudCA9IGludChybmcuaW50ZWdlcnMoMTUsIDQxKSkKICAgICAgICBkZWNsaW5lX3RhcmdldCA9IGZsb2F0KHJuZy51bmlmb3JtKDAuNiwgMC45KSkKICAgICAgICByb3dzLmFwcGVuZCgKICAgICAgICAgICAgewogICAgICAgICAgICAgICAgImVwaXNvZGVfaWQiOiBfZXBpc29kZV9pZChybmcpLAogICAgICAgICAgICAgICAgImVwaXNvZGVfdHlwZSI6ICJjYXJkX3Rlc3RpbmdfYnVyc3QiLAogICAgICAgICAgICAgICAgImFmZmVjdGVkX2VudGl0eV90eXBlIjogInRlcm1pbmFsIiwKICAgICAgICAgICAgICAgICJhZmZlY3RlZF9lbnRpdHlfaWQiOiB0ZXJtX2lkLAogICAgICAgICAgICAgICAgIndpbmRvd19zdGFydCI6IHN0YXJ0LAogICAgICAgICAgICAgICAgIndpbmRvd19lbmQiOiBlbmQsCiAgICAgICAgICAgICAgICAiaW50ZW5zaXR5X3BhcmFtXzEiOiBmbG9hdChyb3dfY291bnQpLAogICAgICAgICAgICAgICAgImludGVuc2l0eV9wYXJhbV8yIjogZGVjbGluZV90YXJnZXQsCiAgICAgICAgICAgICAgICAicm93X2NvdW50X2hpbnQiOiByb3dfY291bnQsCiAgICAgICAgICAgIH0KICAgICAgICApCgogICAgIyAyLiB0ZXJtaW5hbF9jb21wcm9taXNlIC0tIHJlc3RyaWN0ZWQgdG8gdGhlIHRlbGVtZXRyeSB3aW5kb3cgKGxhc3QgMzAgZGF5cykKICAgICMgICAgc28gYm90aCB0aGUgdGFtcGVyLWZsYWcgdGVsZW1ldHJ5IHNpZ25hbCBhbmQgdGhlIGF1dGggZGVjbGluZS1yYXRlCiAgICAjICAgIHNpZ25hbCBjby1vY2N1ci4gQmlhc2VkIDcwJSB0b3dhcmQgc3RhcnRpbmcgb3V0c2lkZSB0cmFkaW5nIGhvdXJzLgogICAgZm9yIF8gaW4gcmFuZ2UoTl9URVJNSU5BTF9D"
    "T01QUk9NSVNFKToKICAgICAgICB0ZXJtX2lkID0gcm5nLmNob2ljZSh0ZXJtaW5hbF9pZHMpCiAgICAgICAgc3RvcmVfaWQgPSB0ZXJtaW5hbF90b19zdG9yZVt0ZXJtX2lkXQogICAgICAgIHByZWZlcl9hZnRlcl9ob3VycyA9IHJuZy5yYW5kb20oKSA8IDAuNwogICAgICAgICMgdHJ5IGEgaGFuZGZ1bCBvZiBjYW5kaWRhdGUgc3RhcnRzLCBrZWVwIHRoZSBmaXJzdCB0aGF0IG1hdGNoZXMgdGhlIGJpYXMKICAgICAgICBzdGFydCA9IE5vbmUKICAgICAgICBmb3IgX2F0dGVtcHQgaW4gcmFuZ2UoOCk6CiAgICAgICAgICAgIGNhbmRpZGF0ZSA9IF9yYW5kb21fdHMocm5nLCBURUxFTUVUUllfV0lORE9XX1NUQVJULCBTSU1fTk9XIC0gcGQuVGltZWRlbHRhKGhvdXJzPTYpKQogICAgICAgICAgICBpbl9ob3VycyA9IGlzX3RyYWRpbmdfaG91cnMoc3RvcmVfaWQsIGNhbmRpZGF0ZSwgYXJjaGV0eXBlX21hcCkKICAgICAgICAgICAgaWYgcHJlZmVyX2FmdGVyX2hvdXJzIGFuZCBub3QgaW5faG91cnM6CiAgICAgICAgICAgICAgICBzdGFydCA9IGNhbmRpZGF0ZQogICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgaWYgbm90IHByZWZlcl9hZnRlcl9ob3VycyBhbmQgaW5faG91cnM6CiAgICAgICAgICAgICAgICBzdGFydCA9IGNhbmRpZGF0ZQogICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICBpZiBzdGFydCBpcyBOb25lOgogICAgICAgICAgICBzdGFydCA9IF9yYW5kb21fdHMocm5nLCBURUxFTUVUUllfV0lORE9XX1NUQVJULCBTSU1fTk9XIC0gcGQuVGltZWRlbHRhKGhvdXJzPTYpKQogICAgICAgIGR1cmF0aW9uX2hvdXJzID0gZmxvYXQocm5nLnVuaWZvcm0oMiwgNikpCiAgICAgICAgZW5kID0gc3RhcnQgKyBwZC5UaW1lZGVsdGEoaG91cnM9ZHVyYXRpb25faG91cnMpCiAgICAgICAgZGVjbGluZV9zdGVwID0gZmxvYXQocm5nLnVuaWZvcm0oMC4yMCwgMC40MCkpCiAgICAgICAgcm93cy5hcHBlbmQoCiAgICAgICAgICAgIHsKICAgICAgICAgICAgICAgICJlcGlzb2RlX2lkIjogX2VwaXNvZGVfaWQocm5nKSwKICAgICAgICAgICAgICAgICJlcGlzb2RlX3R5cGUiOiAidGVybWluYWxfY29tcHJvbWlzZSIsCiAgICAgICAgICAgICAgICAiYWZmZWN0ZWRfZW50aXR5X3R5cGUiOiAidGVybWluYWwiLAogICAgICAgICAgICAgICAgImFmZmVjdGVkX2VudGl0eV9pZCI6IHRlcm1faWQsCiAgICAgICAgICAgICAgICAid2luZG93X3N0YXJ0Ijogc3RhcnQsCiAgICAgICAgICAgICAgICAid2luZG93X2VuZCI6IGVuZCwKICAgICAgICAgICAgICAgICJpbnRlbnNpdHlfcGFyYW1fMSI6IGRlY2xpbmVfc3RlcCwKICAgICAgICAgICAgICAgICJpbnRlbnNpdHlfcGFyYW1fMiI6IGR1cmF0aW9uX2hvdXJzLAogICAgICAgICAgICAgICAgInJvd19jb3VudF9oaW50IjogMCwKICAgICAgICAgICAgfQogICAgICAgICkKCiAgICAjIDMuIGlzc3Vlcl9kZWdyYWRhdGlvbiAtLSBhbnkgdGltZSBpbiB0aGUgOTAtZGF5IGF1dGggd2luZG93CiAgICBmb3IgXyBpbiByYW5nZShOX0lTU1VFUl9ERUdSQURBVElPTik6CiAgICAgICAgaXNzdWVyX2JpbiA9IHJuZy5jaG9pY2UoaXNzdWVyX2JpbnMpCiAgICAgICAgc3RhcnQgPSBfcmFuZG9tX3RzKHJuZywgQVVUSF9XSU5ET1dfU1RBUlQsIFNJTV9OT1cgLSBwZC5UaW1lZGVsdGEoaG91cnM9OCkpCiAgICAgICAgZHVyYXRpb25faG91cnMgPSBmbG9hdChybmcudW5pZm9ybSgzLCA4KSkKICAgICAgICBlbmQgPSBzdGFydCArIHBkLlRpbWVkZWx0YShob3Vycz1kdXJhdGlvbl9ob3VycykKICAgICAgICBiYXNlbGluZV9hcHByb3ZhbCA9IGZsb2F0KHJuZy51bmlmb3JtKDAuODUsIDAuOTUpKQogICAgICAgIGRyb3BfcGN0ID0gZmxvYXQocm5nLnVuaWZvcm0oMC4xNSwgMC4zNSkpCiAgICAgICAgcm93cy5hcHBlbmQoCiAgICAgICAgICAgIHsKICAgICAgICAgICAgICAgICJlcGlzb2RlX2lkIjogX2VwaXNvZGVfaWQocm5nKSwKICAgICAgICAgICAgICAgICJlcGlzb2RlX3R5cGUiOiAiaXNzdWVyX2RlZ3JhZGF0aW9uIiwKICAgICAgICAgICAgICAgICJhZmZlY3RlZF9lbnRpdHlfdHlwZSI6ICJpc3N1ZXIiLAogICAgICAgICAgICAgICAgImFmZmVjdGVkX2VudGl0eV9pZCI6IGlzc3Vlcl9iaW4sCiAgICAgICAgICAgICAgICAid2luZG93X3N0YXJ0Ijogc3RhcnQsCiAgICAgICAgICAgICAgICAid2luZG93X2VuZCI6IGVuZCwKICAgICAgICAgICAgICAgICJpbnRlbnNpdHlfcGFyYW1fMSI6IGJhc2VsaW5lX2FwcHJvdmFsLAogICAgICAgICAgICAgICAgImludGVuc2l0eV9wYXJhbV8yIjogZHJvcF9wY3QsCiAgICAgICAgICAgICAgICAicm93X2NvdW50X2hpbnQiOiAwLAogICAgICAgICAgICB9CiAgICAgICAgKQoKICAgICMgNC4gdGVybWluYWxfZGFya19vdXRhZ2UgLS0gbXVzdCBmYWxsIGVudGlyZWx5IGluc2lkZSBhIHRyYWRpbmctaG91cnMKICAgICMgICAgc2Vzc2lvbiBmb3IgdGhhdCBzdG9yZSdzIGFyY2hldHlwZSwgd2l0aGluIHRoZSB0ZWxlbWV0cnkgd2luZG93LgogICAgZm9yIF8gaW4gcmFuZ2UoTl9URVJNSU5BTF9EQVJLKToKICAgICAgICB0ZXJtX2lkID0gcm5nLmNob2ljZSh0ZXJtaW5hbF9pZHMpCiAgICAgICAgc3RvcmVfaWQgPSB0ZXJtaW5hbF90b19zdG9yZVt0ZXJtX2lkXQogICAgICAgIGR1cmF0aW9uX21pbiA9IGludChybmcuaW50ZWdlcnMoMjAsIDkxKSkKICAgICAgICBzdGFydCA9IE5vbmUKICAgICAgICBmb3IgX2F0dGVtcHQgaW4gcmFuZ2UoMTIpOgogICAgICAgICAgICBjYW5kaWRhdGUgPSBfcmFuZG9tX3RzKHJuZywgVEVMRU1FVFJZX1dJTkRPV19TVEFSVCwgU0lNX05PVyAtIHBkLlRpbWVkZWx0YShob3Vycz0yKSkKICAgICAgICAgICAgY2FuZGlkYXRlX2VuZCA9IGNhbmRpZGF0ZSArIHBkLlRpbWVkZWx0YShtaW51dGVzPWR1cmF0aW9uX21pbikKICAgICAgICAgICAgaWYgaXNfdHJhZGluZ19ob3VycyhzdG9yZV9pZCwgY2FuZGlkYXRlLCBhcmNoZXR5cGVfbWFwKSBhbmQgaXNfdHJhZGluZ19ob3Vy"
    "cygKICAgICAgICAgICAgICAgIHN0b3JlX2lkLCBjYW5kaWRhdGVfZW5kLCBhcmNoZXR5cGVfbWFwCiAgICAgICAgICAgICk6CiAgICAgICAgICAgICAgICBzdGFydCA9IGNhbmRpZGF0ZQogICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICBpZiBzdGFydCBpcyBOb25lOgogICAgICAgICAgICAjIGZhbGwgYmFjayB0byBhIHN0YW5kYXJkLWhvdXJzIHdpbmRvdyAoMTE6MDAgbG9jYWwtbmFpdmUgVVRDKSBvbiBhCiAgICAgICAgICAgICMgZGV0ZXJtaW5pc3RpYyBkYXkgb2Zmc2V0IHNvIHRoZSBlcGlzb2RlIGlzIG5ldmVyIHNpbGVudGx5IGRyb3BwZWQKICAgICAgICAgICAgZGF5X29mZnNldCA9IGludChybmcuaW50ZWdlcnMoMCwgVEVMRU1FVFJZX0JBQ0tGSUxMX0RBWVMgLSAxKSkKICAgICAgICAgICAgc3RhcnQgPSBURUxFTUVUUllfV0lORE9XX1NUQVJUICsgcGQuVGltZWRlbHRhKGRheXM9ZGF5X29mZnNldCwgaG91cnM9MTEpCiAgICAgICAgZW5kID0gc3RhcnQgKyBwZC5UaW1lZGVsdGEobWludXRlcz1kdXJhdGlvbl9taW4pCiAgICAgICAgcm93cy5hcHBlbmQoCiAgICAgICAgICAgIHsKICAgICAgICAgICAgICAgICJlcGlzb2RlX2lkIjogX2VwaXNvZGVfaWQocm5nKSwKICAgICAgICAgICAgICAgICJlcGlzb2RlX3R5cGUiOiAidGVybWluYWxfZGFya19vdXRhZ2UiLAogICAgICAgICAgICAgICAgImFmZmVjdGVkX2VudGl0eV90eXBlIjogInRlcm1pbmFsIiwKICAgICAgICAgICAgICAgICJhZmZlY3RlZF9lbnRpdHlfaWQiOiB0ZXJtX2lkLAogICAgICAgICAgICAgICAgIndpbmRvd19zdGFydCI6IHN0YXJ0LAogICAgICAgICAgICAgICAgIndpbmRvd19lbmQiOiBlbmQsCiAgICAgICAgICAgICAgICAiaW50ZW5zaXR5X3BhcmFtXzEiOiBmbG9hdChkdXJhdGlvbl9taW4pLAogICAgICAgICAgICAgICAgImludGVuc2l0eV9wYXJhbV8yIjogMC4wLAogICAgICAgICAgICAgICAgInJvd19jb3VudF9oaW50IjogMCwKICAgICAgICAgICAgfQogICAgICAgICkKCiAgICBncm91bmRfdHJ1dGggPSBwZC5EYXRhRnJhbWUocm93cykKICAgICMgZmxvb3IgdG8gbWljcm9zZWNvbmQgcmVzb2x1dGlvbiBoZXJlIHNvIG5zLXByZWNpc2lvbiBub2lzZSBmcm9tCiAgICAjIHBkLlRpbWVkZWx0YShzZWNvbmRzPTxmbG9hdD4pIG5ldmVyIGxlYWtzIGludG8gbGF0ZXIgYXJpdGhtZXRpYyAoZS5nLgogICAgIyBucC5kYXRldGltZTY0KHdpbmRvd19zdGFydCkgKyB0aW1lZGVsdGE2NFt1c10gaW4gYXV0aF9ldmVudHMucHkncwogICAgIyBjYXJkX3Rlc3RpbmdfYnVyc3Qgc3ViLWdlbmVyYXRpb24sIHdoaWNoIHdvdWxkIG90aGVyd2lzZSBzaWxlbnRseQogICAgIyB1cGNhc3QgdGhlIHJlc3VsdCB0byBkYXRldGltZTY0W25zXSBhbmQgZmFpbCB0aGUgc2NoZW1hIGNhc3QgYXQgd3JpdGUgdGltZSkKICAgIGdyb3VuZF90cnV0aFsid2luZG93X3N0YXJ0Il0gPSBwZC50b19kYXRldGltZShncm91bmRfdHJ1dGhbIndpbmRvd19zdGFydCJdKS5kdC5mbG9vcigidXMiKQogICAgZ3JvdW5kX3RydXRoWyJ3aW5kb3dfZW5kIl0gPSBwZC50b19kYXRldGltZShncm91bmRfdHJ1dGhbIndpbmRvd19lbmQiXSkuZHQuZmxvb3IoInVzIikKICAgIHJldHVybiBncm91bmRfdHJ1dGgK"
)
_FILES["episodes.py"] = _b64_episodes_py
_b64_auth_events_py = (
    "..."
)
_FILES["auth_events.py"] = _b64_auth_events_py
_b64_telemetry_py = (
    "IiIidGVybWluYWxfdGVsZW1ldHJ5IHJhdyBzdHJlYW0gZ2VuZXJhdG9yLiBTZWUgR0VORVJBVE9SX1NQRUMubWQgc2VjdGlvbnMgMiBhbmQgNC4yLgoKMTUtbWludXRlIGhlYXJ0YmVhdCB4IDEsNTAwIHRlcm1pbmFscyB4IDMwIGRheXMgPSA0LDMyMCwwMDAgY2FuZGlkYXRlIHJvd3MsIG1pbnVzCnRoZSByb3dzIHJlbW92ZWQgaW5zaWRlIHRlcm1pbmFsX2Rhcmtfb3V0YWdlIGVwaXNvZGUgd2luZG93cyAodGhlIEFCU0VOQ0Ugb2YgYQpoZWFydGJlYXQgaXMgdGhlIHNpZ25hbCByZWZsZXggMyBkZXRlY3RzIC0tIG5vIHJvdyBpcyBlbWl0dGVkLCBub3QgYSBmbGFnKS4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAoKZnJvbSAuY29yZSBpbXBvcnQgcm5nX2ZvciwgU0lNX05PVywgVEVMRU1FVFJZX0JBQ0tGSUxMX0RBWVMKClRFTEVNRVRSWV9XSU5ET1dfU1RBUlQgPSBTSU1fTk9XIC0gcGQuVGltZWRlbHRhKGRheXM9VEVMRU1FVFJZX0JBQ0tGSUxMX0RBWVMpCkhFQVJUQkVBVF9JTlRFUlZBTF9NSU4gPSAxNQpJTlRFUlZBTFNfUEVSX0RBWSA9ICgyNCAqIDYwKSAvLyBIRUFSVEJFQVRfSU5URVJWQUxfTUlOICAjIDk2CgoKZGVmIGdlbmVyYXRlX3RlbGVtZXRyeShkaW1fdGVybWluYWwsIGdyb3VuZF90cnV0aCwgbWFzdGVyX3NlZWQ9Tm9uZSwgd2luZG93X3N0YXJ0PU5vbmUsIHdpbmRvd19lbmQ9Tm9uZSk6CiAgICAiIiJ3aW5kb3dfc3RhcnQvd2luZG93X2VuZCBkZWZhdWx0IHRvIHRoZSBmdWxsIDMwLWRheSBiYWNrZmlsbCB3aW5kb3cgLS0KICAgIGJhY2tmaWxsLnB5J3MgY2FsbCBpcyB1bmNoYW5nZWQuIGxpdmVfcmVwbGF5LnB5IHBhc3NlcyBhIHNob3J0IHRyYWlsaW5nCiAgICB3aW5kb3cgZGlyZWN0bHkgKG5vdCBhIHNsaWNlIG9mIGEgZnVsbCAzMC1kYXkgZ2VuZXJhdGlvbiksIHNvIHRoZSBoZWFydGJlYXQKICAgIGdyaWQgYnVpbHQgaGVyZSBpcyBzaXplZCB0byB0aGF0IHdpbmRvdywgbm90IHRoZSBmdWxsIGJhY2tmaWxsLiIiIgogICAgcm5nID0gcm5nX2ZvcigidGVsZW1ldHJ5IiwgbWFzdGVyX3NlZWQpIGlmIG1hc3Rlcl9zZWVkIGlzIG5vdCBOb25lIGVsc2Ugcm5nX2ZvcigidGVsZW1ldHJ5IikKCiAgICBpZiB3aW5kb3dfc3RhcnQgaXMgTm9uZToKICAgICAgICB3aW5kb3dfc3RhcnQgPSBURUxFTUVUUllfV0lORE9XX1NUQVJUCiAgICBpZiB3aW5kb3dfZW5kIGlzIE5vbmU6CiAgICAgICAgd2luZG93X2VuZCA9IFNJTV9OT1cKCiAgICBuX3Rlcm1pbmFscyA9IGxlbihkaW1fdGVybWluYWwpCiAgICB3aW5kb3dfbWludXRlcyA9IG1heChpbnQoKHdpbmRvd19lbmQgLSB3aW5kb3dfc3RhcnQpLnRvdGFsX3NlY29uZHMoKSAvLyA2MCksIDApCiAgICBuX2ludGVydmFscyA9IG1heCh3aW5kb3dfbWludXRlcyAvLyBIRUFSVEJFQVRfSU5URVJWQUxfTUlOLCAxKQoKICAgIHRlcm1pbmFsX2lkcyA9IGRpbV90ZXJtaW5hbFsidGVybWluYWxfaWQiXS50b19udW1weSgpCiAgICBzdG9yZV9pZHMgPSBkaW1fdGVybWluYWxbInN0b3JlX2lkIl0udG9fbnVtcHkoKQogICAgbWVyY2hhbnRfaWRzID0gZGltX3Rlcm1pbmFsWyJtZXJjaGFudF9pZCJdLnRvX251bXB5KCkKCiAgICBvZmZzZXRzX21pbiA9IG5wLmFyYW5nZShuX2ludGVydmFscykgKiBIRUFSVEJFQVRfSU5URVJWQUxfTUlOCiAgICBiYXNlX3RpbWVzID0gbnAuZGF0ZXRpbWU2NCh3aW5kb3dfc3RhcnQpICsgb2Zmc2V0c19taW4uYXN0eXBlKCJ0aW1lZGVsdGE2NFttXSIpCgogICAgIyBmdWxsIHRlcm1pbmFsIHggaW50ZXJ2YWwgZ3JpZCwgYnVpbHQgb25jZSB2aWEgcmVwZWF0L3RpbGUgKHZlY3Rvcml6ZWQsIG5vIHB5dGhvbiBsb29wKQogICAgZXZlbnRfdGltZSA9IG5wLnRpbGUoYmFzZV90aW1lcywgbl90ZXJtaW5hbHMpCiAgICB0ZXJtaW5hbF9pZF9jb2wgPSBucC5yZXBlYXQodGVybWluYWxfaWRzLCBuX2ludGVydmFscykKICAgIHN0b3JlX2lkX2NvbCA9IG5wLnJlcGVhdChzdG9yZV9pZHMsIG5faW50ZXJ2YWxzKQogICAgbWVyY2hhbnRfaWRfY29sID0gbnAucmVwZWF0KG1lcmNoYW50X2lkcywgbl9pbnRlcnZhbHMpCgogICAgbl9yb3dzID0gbGVuKGV2ZW50X3RpbWUpCiAgICBkZiA9IHBkLkRhdGFGcmFtZSgKICAgICAgICB7CiAgICAgICAgICAgICJldmVudF90aW1lIjogZXZlbnRfdGltZSwKICAgICAgICAgICAgInRlcm1pbmFsX2lkIjogdGVybWluYWxfaWRfY29sLAogICAgICAgICAgICAic3RvcmVfaWQiOiBzdG9yZV9pZF9jb2wsCiAgICAgICAgICAgICJtZXJjaGFudF9pZCI6IG1lcmNoYW50X2lkX2NvbCwKICAgICAgICB9CiAgICApCgogICAgIyByZW1vdmUgcm93cyBpbnNpZGUgdGVybWluYWxfZGFya19vdXRhZ2Ugd2luZG93cyAtLSB0aGUgYWJzZW5jZSBJUyB0aGUgc2lnbmFsCiAgICBpZiBncm91bmRfdHJ1dGggaXMgbm90IE5vbmUgYW5kIGxlbihncm91bmRfdHJ1dGgpOgogICAgICAgIG91dGFnZXMgPSBncm91bmRfdHJ1dGhbZ3JvdW5kX3RydXRoWyJlcGlzb2RlX3R5cGUiXSA9PSAidGVybWluYWxfZGFya19vdXRhZ2UiXQogICAgICAgIGRyb3BfbWFzayA9IG5wLnplcm9zKG5fcm93cywgZHR5cGU9Ym9vbCkKICAgICAgICBmb3IgXywgZXAgaW4gb3V0YWdlcy5pdGVycm93cygpOgogICAgICAgICAgICB0ZXJtX21hc2sgPSBkZlsidGVybWluYWxfaWQiXS50b19udW1weSgpID09IGVwWyJhZmZlY3RlZF9lbnRpdHlfaWQiXQogICAgICAgICAgICB0aW1lX21hc2sgPSAoZGZbImV2ZW50X3RpbWUiXSA+PSBlcFsid2luZG93X3N0YXJ0Il0pICYgKGRmWyJldmVudF90aW1lIl0gPD0gZXBbIndpbmRvd19lbmQiXSkKICAgICAgICAgICAgZHJvcF9tYXNrIHw9IHRlcm1fbWFzayAmIHRpbWVfbWFzay50b19udW1weSgpCiAgICAgICAgZGYgPSBkZlt+ZHJvcF9tYXNrXS5yZXNldF9pbmRleChkcm9wPVRydWUpCiAgICAgICAgbl9yb3dzID0gbGVuKGRmKQoKICAgICAgICAjIHRhbXBlcl9mbGFn"
    "PVRydWUgZm9yIHJvd3MgaW5zaWRlIHRlcm1pbmFsX2NvbXByb21pc2Ugd2luZG93cwogICAgICAgIHRhbXBlcl9tYXNrID0gbnAuemVyb3Mobl9yb3dzLCBkdHlwZT1ib29sKQogICAgICAgIGNvbXByb21pc2VzID0gZ3JvdW5kX3RydXRoW2dyb3VuZF90cnV0aFsiZXBpc29kZV90eXBlIl0gPT0gInRlcm1pbmFsX2NvbXByb21pc2UiXQogICAgICAgIGZvciBfLCBlcCBpbiBjb21wcm9taXNlcy5pdGVycm93cygpOgogICAgICAgICAgICB0ZXJtX21hc2sgPSBkZlsidGVybWluYWxfaWQiXS50b19udW1weSgpID09IGVwWyJhZmZlY3RlZF9lbnRpdHlfaWQiXQogICAgICAgICAgICB0aW1lX21hc2sgPSAoZGZbImV2ZW50X3RpbWUiXSA+PSBlcFsid2luZG93X3N0YXJ0Il0pICYgKGRmWyJldmVudF90aW1lIl0gPD0gZXBbIndpbmRvd19lbmQiXSkKICAgICAgICAgICAgdGFtcGVyX21hc2sgfD0gdGVybV9tYXNrICYgdGltZV9tYXNrLnRvX251bXB5KCkKICAgIGVsc2U6CiAgICAgICAgdGFtcGVyX21hc2sgPSBucC56ZXJvcyhuX3Jvd3MsIGR0eXBlPWJvb2wpCgogICAgZGZbInRhbXBlcl9mbGFnIl0gPSB0YW1wZXJfbWFzawogICAgZGZbImluZ2VzdF90aW1lIl0gPSBkZlsiZXZlbnRfdGltZSJdICAjIHBlcnR1cmJlZCBsYXRlciBieSBkcV9mYXVsdHMucHkKICAgIGRmWyJoZWFydGJlYXRfb2siXSA9IHJuZy5yYW5kb20obl9yb3dzKSA+IDAuMDEgICMgfjElIGNvc21ldGljIGhpY2N1cCwgaW5kZXBlbmRlbnQgb2YgdGFtcGVyCiAgICBkZlsiYmF0dGVyeV9wY3QiXSA9IG5wLmNsaXAocm5nLm5vcm1hbCg3MCwgMTUsIHNpemU9bl9yb3dzKSwgNSwgMTAwKS5yb3VuZCgxKQogICAgZGZbInNpZ25hbF9zdHJlbmd0aCJdID0gbnAuY2xpcChybmcubm9ybWFsKDc1LCAxMiwgc2l6ZT1uX3Jvd3MpLCAxMCwgMTAwKS5yb3VuZCgxKQoKICAgIGRmWyJ0ZWxlbWV0cnlfaWQiXSA9IFtmIlRFTC17eDowMTB4fSIgZm9yIHggaW4gcm5nLmludGVnZXJzKDAsIDIqKjQwLCBzaXplPW5fcm93cyldCiAgICBkZlsic2NoZW1hX3ZlcnNpb24iXSA9IG5wLmludDMyKDEpICAjIHRlbGVtZXRyeSBoYXMgbm8gc2NoZW1hLWV2b2x1dGlvbiBmaWVsZDsga2VwdCBmb3IgY29uc2lzdGVuY3kKICAgIGRmWyJldmVudF90aW1lIl0gPSBwZC50b19kYXRldGltZShkZlsiZXZlbnRfdGltZSJdKQogICAgZGZbImluZ2VzdF90aW1lIl0gPSBwZC50b19kYXRldGltZShkZlsiaW5nZXN0X3RpbWUiXSkKCiAgICBjb2x1bW5fb3JkZXIgPSBbCiAgICAgICAgInRlbGVtZXRyeV9pZCIsICJldmVudF90aW1lIiwgImluZ2VzdF90aW1lIiwgInRlcm1pbmFsX2lkIiwgInN0b3JlX2lkIiwKICAgICAgICAibWVyY2hhbnRfaWQiLCAiaGVhcnRiZWF0X29rIiwgInRhbXBlcl9mbGFnIiwgImJhdHRlcnlfcGN0IiwKICAgICAgICAic2lnbmFsX3N0cmVuZ3RoIiwgInNjaGVtYV92ZXJzaW9uIiwKICAgIF0KICAgIGRmID0gZGZbY29sdW1uX29yZGVyXS5zb3J0X3ZhbHVlcyhbImV2ZW50X3RpbWUiLCAidGVybWluYWxfaWQiXSkucmVzZXRfaW5kZXgoZHJvcD1UcnVlKQogICAgcmV0dXJuIGRmCg=="
)
_FILES["telemetry.py"] = _b64_telemetry_py
_b64_disputes_py = (
    "IiIiZGlzcHV0ZV9ldmVudHMgcmF3IHN0cmVhbSBnZW5lcmF0b3IuIFNlZSBHRU5FUkFUT1JfU1BFQy5tZCBzZWN0aW9ucyA0LjMgYW5kIDUncwptYXR1cml0eS1jb2hvcnQgbm90ZS4KCkEgY2FuZGlkYXRlIGF1dGggYmVjb21lcyBhIGRpc3B1dGUgcm93IG9ubHkgaWYgaXRzIGNvbXB1dGVkIGRpc3B1dGVfdGltZQooZXZlbnRfdGltZSArIFVuaWZvcm0oMzAsIDYwKSBkYXlzKSBmYWxscyBhdCBvciBiZWZvcmUgU0lNX05PVy4gQXV0aHMgaW5zaWRlIHRoZQpULTMwIC0+IFQgd2luZG93IG1vc3RseSBmYWlsIHRoYXQgdGVzdCBhbmQgc2ltcGx5IHByb2R1Y2Ugbm8gZGlzcHV0ZSByb3cgeWV0IC0tCndoaWNoIGlzIGV4YWN0bHkgdGhlICJhd2FpdGluZyBncmFkaW5nIiBjb2hvcnQgdGhlIGNvbnRyYWN0IGRlc2NyaWJlcywgbW9kZWxsZWQKc3RydWN0dXJhbGx5IHJhdGhlciB0aGFuIGZsYWdnZWQgYWZ0ZXIgdGhlIGZhY3QuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKCmZyb20gLmNvcmUgaW1wb3J0IHJuZ19mb3IsIFNJTV9OT1cKCkRJU1BVVEVfUkFURSA9IDAuMDAzICAjIH4wLjMlIG9mIHByaW1hcnkgJ2F1dGgnIHJvd3MsIGNvbnRyYWN0IHNlY3Rpb24gNQpGUkFVRF9MSU5LRURfU0hBUkUgPSAwLjcwICAjIHNoYXJlIG9mIHRoZSBkaXNwdXRlIHRhcmdldCBwcmVmZXJlbnRpYWxseSBkcmF3biBmcm9tIGVwaXNvZGUtbGlua2VkIGF1dGhzCgpESVNQVVRFX1JFQVNPTl9CQVNFTElORSA9IFsibm90X2FzX2Rlc2NyaWJlZCIsICJkdXBsaWNhdGVfcHJvY2Vzc2luZyIsICJvdGhlciIsICJmcmF1ZCJdCkRJU1BVVEVfUkVBU09OX0JBU0VMSU5FX1dFSUdIVFMgPSBbMC4zNSwgMC4yMCwgMC4yNSwgMC4yMF0KCgpkZWYgZ2VuZXJhdGVfZGlzcHV0ZXMoYXV0aF9ldmVudHM6IHBkLkRhdGFGcmFtZSwgZ3JvdW5kX3RydXRoOiBwZC5EYXRhRnJhbWUsIG1hc3Rlcl9zZWVkPU5vbmUpOgogICAgcm5nID0gcm5nX2ZvcigiZGlzcHV0ZXMiLCBtYXN0ZXJfc2VlZCkgaWYgbWFzdGVyX3NlZWQgaXMgbm90IE5vbmUgZWxzZSBybmdfZm9yKCJkaXNwdXRlcyIpCgogICAgY2FuZGlkYXRlcyA9IGF1dGhfZXZlbnRzWwogICAgICAgIChhdXRoX2V2ZW50c1siZXZlbnRfdHlwZSJdID09ICJhdXRoIikgJiAoYXV0aF9ldmVudHNbImF1dGhfcmVzdWx0Il0gPT0gImFwcHJvdmVkIikKICAgIF0ucmVzZXRfaW5kZXgoZHJvcD1UcnVlKQogICAgbl9jYW5kaWRhdGVzID0gbGVuKGNhbmRpZGF0ZXMpCiAgICB0YXJnZXRfbiA9IGludChyb3VuZChESVNQVVRFX1JBVEUgKiBuX2NhbmRpZGF0ZXMpKQoKICAgIGVwaXNvZGVfbGlua2VkX21hc2sgPSBucC56ZXJvcyhuX2NhbmRpZGF0ZXMsIGR0eXBlPWJvb2wpCiAgICBpZiBncm91bmRfdHJ1dGggaXMgbm90IE5vbmUgYW5kIGxlbihncm91bmRfdHJ1dGgpOgogICAgICAgIHJpc2t5ID0gZ3JvdW5kX3RydXRoW2dyb3VuZF90cnV0aFsiZXBpc29kZV90eXBlIl0uaXNpbihbImNhcmRfdGVzdGluZ19idXJzdCIsICJ0ZXJtaW5hbF9jb21wcm9taXNlIl0pXQogICAgICAgIGNhbmRfdGVybSA9IGNhbmRpZGF0ZXNbInRlcm1pbmFsX2lkIl0udG9fbnVtcHkoKQogICAgICAgIGNhbmRfdGltZSA9IGNhbmRpZGF0ZXNbImV2ZW50X3RpbWUiXS50b19udW1weSgpCiAgICAgICAgZm9yIF8sIGVwIGluIHJpc2t5Lml0ZXJyb3dzKCk6CiAgICAgICAgICAgIHRlcm1fbWFzayA9IGNhbmRfdGVybSA9PSBlcFsiYWZmZWN0ZWRfZW50aXR5X2lkIl0KICAgICAgICAgICAgdGltZV9tYXNrID0gKGNhbmRfdGltZSA+PSBucC5kYXRldGltZTY0KGVwWyJ3aW5kb3dfc3RhcnQiXSkpICYgKAogICAgICAgICAgICAgICAgY2FuZF90aW1lIDw9IG5wLmRhdGV0aW1lNjQoZXBbIndpbmRvd19lbmQiXSkKICAgICAgICAgICAgKQogICAgICAgICAgICBlcGlzb2RlX2xpbmtlZF9tYXNrIHw9IHRlcm1fbWFzayAmIHRpbWVfbWFzawoKICAgIGxpbmtlZF9pZHggPSBucC5mbGF0bm9uemVybyhlcGlzb2RlX2xpbmtlZF9tYXNrKQogICAgdW5saW5rZWRfaWR4ID0gbnAuZmxhdG5vbnplcm8ofmVwaXNvZGVfbGlua2VkX21hc2spCgogICAgbl9mcm9tX2xpbmtlZCA9IG1pbihpbnQodGFyZ2V0X24gKiBGUkFVRF9MSU5LRURfU0hBUkUpLCBsZW4obGlua2VkX2lkeCkpCiAgICBuX2Zyb21fdW5saW5rZWQgPSB0YXJnZXRfbiAtIG5fZnJvbV9saW5rZWQKCiAgICBjaG9zZW5fbGlua2VkID0gcm5nLmNob2ljZShsaW5rZWRfaWR4LCBzaXplPW5fZnJvbV9saW5rZWQsIHJlcGxhY2U9RmFsc2UpIGlmIG5fZnJvbV9saW5rZWQgZWxzZSBucC5lbXB0eSgwLCBkdHlwZT1pbnQpCiAgICBjaG9zZW5fdW5saW5rZWQgPSAoCiAgICAgICAgcm5nLmNob2ljZSh1bmxpbmtlZF9pZHgsIHNpemU9bWluKG5fZnJvbV91bmxpbmtlZCwgbGVuKHVubGlua2VkX2lkeCkpLCByZXBsYWNlPUZhbHNlKQogICAgICAgIGlmIG5fZnJvbV91bmxpbmtlZCA+IDAKICAgICAgICBlbHNlIG5wLmVtcHR5KDAsIGR0eXBlPWludCkKICAgICkKICAgIGNob3Nlbl9pZHggPSBucC5jb25jYXRlbmF0ZShbY2hvc2VuX2xpbmtlZCwgY2hvc2VuX3VubGlua2VkXSkuYXN0eXBlKGludCkKCiAgICBjaG9zZW4gPSBjYW5kaWRhdGVzLmlsb2NbY2hvc2VuX2lkeF0ucmVzZXRfaW5kZXgoZHJvcD1UcnVlKQogICAgbl9jaG9zZW4gPSBsZW4oY2hvc2VuKQoKICAgIGxhZ19kYXlzID0gcm5nLnVuaWZvcm0oMzAsIDYwLCBzaXplPW5fY2hvc2VuKQogICAgZXZlbnRfdGltZV9ucCA9IHBkLnRvX2RhdGV0aW1lKGNob3NlblsiZXZlbnRfdGltZSJdKS50b19udW1weSgpCiAgICBkaXNwdXRlX3RpbWUgPSBldmVudF90aW1lX25wICsgKGxhZ19kYXlzICogODY0MDAgKiAxZTYpLmFzdHlwZSgidGltZWRlbHRhNjRbdXNdIikKCiAgICBtYXR1cmVkX21hc2sgPSBkaXNwdXRlX3RpbWUgPD0gbnAuZGF0ZXRpbWU2NChTSU1fTk9XKQogICAgY2hvc2VuID0gY2hvc2Vu"
    "W21hdHVyZWRfbWFza10ucmVzZXRfaW5kZXgoZHJvcD1UcnVlKQogICAgZGlzcHV0ZV90aW1lID0gZGlzcHV0ZV90aW1lW21hdHVyZWRfbWFza10KICAgIGlzX2xpbmtlZCA9IG5wLmNvbmNhdGVuYXRlKAogICAgICAgIFtucC5vbmVzKG5fZnJvbV9saW5rZWQsIGR0eXBlPWJvb2wpLCBucC56ZXJvcyhsZW4oY2hvc2VuX3VubGlua2VkKSwgZHR5cGU9Ym9vbCldCiAgICApW21hdHVyZWRfbWFza10gaWYgbl9jaG9zZW4gZWxzZSBucC5hcnJheShbXSwgZHR5cGU9Ym9vbCkKICAgIG5fZmluYWwgPSBsZW4oY2hvc2VuKQoKICAgIHJlYXNvbiA9IG5wLndoZXJlKAogICAgICAgIGlzX2xpbmtlZCwKICAgICAgICAiZnJhdWQiLAogICAgICAgIHJuZy5jaG9pY2UoRElTUFVURV9SRUFTT05fQkFTRUxJTkUsIHNpemU9bl9maW5hbCwgcD1ESVNQVVRFX1JFQVNPTl9CQVNFTElORV9XRUlHSFRTKSwKICAgICkKICAgIG91dGNvbWVfcm9sbCA9IHJuZy5yYW5kb20obl9maW5hbCkKICAgIGNhcmRob2xkZXJfd2luX3Byb2IgPSBucC53aGVyZShyZWFzb24gPT0gImZyYXVkIiwgMC42NSwgMC40MCkKICAgIG91dGNvbWUgPSBucC53aGVyZShvdXRjb21lX3JvbGwgPCBjYXJkaG9sZGVyX3dpbl9wcm9iLCAiY2FyZGhvbGRlcl93b24iLCAibWVyY2hhbnRfd29uIikKCiAgICBkZiA9IHBkLkRhdGFGcmFtZSgKICAgICAgICB7CiAgICAgICAgICAgICJkaXNwdXRlX2lkIjogW2YiRFNQLXt4OjAxMHh9IiBmb3IgeCBpbiBybmcuaW50ZWdlcnMoMCwgMioqNDAsIHNpemU9bl9maW5hbCldLAogICAgICAgICAgICAiYXV0aF9pZCI6IGNob3NlblsiYXV0aF9pZCJdLnRvX251bXB5KCksCiAgICAgICAgICAgICJkaXNwdXRlX3RpbWUiOiBwZC50b19kYXRldGltZShkaXNwdXRlX3RpbWUpLAogICAgICAgICAgICAiZGlzcHV0ZV9yZWFzb24iOiByZWFzb24sCiAgICAgICAgICAgICJkaXNwdXRlX291dGNvbWUiOiBvdXRjb21lLAogICAgICAgICAgICAiYW1vdW50IjogY2hvc2VuWyJhbW91bnQiXS50b19udW1weSgpLAogICAgICAgIH0KICAgICkKICAgIHJldHVybiBkZi5zb3J0X3ZhbHVlcyhbImRpc3B1dGVfdGltZSIsICJkaXNwdXRlX2lkIl0pLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkK"
)
_FILES["disputes.py"] = _b64_disputes_py
_b64_dq_faults_py = (
    "IiIiVGhlIHNpeCBzdHJlYW0gRFEgZmF1bHRzLiBTZWUgR0VORVJBVE9SX1NQRUMubWQgc2VjdGlvbiA2LgoKQXBwbGllZCB0byBhbHJlYWR5LWdlbmVyYXRlZCByYXcgZnJhbWVzIChhdXRoX2V2ZW50cywgdGVybWluYWxfdGVsZW1ldHJ5KS4gRWFjaApmdW5jdGlvbiByZXR1cm5zIHRoZSBtdXRhdGVkIGZyYW1lIHBsdXMgYSBzbWFsbCBkaWN0IG9mIGNvdW50cyBmb3IgdGhlCkxhbmRpbmdfTWFuaWZlc3QuY3N2IGZhdWx0IHJlZ2lzdGVyIC0tIGEgcnVsZSByZXBvcnRpbmcgemVybyBmYXVsdHMgaW5qZWN0ZWQKd291bGQgYmUgYSBidWcgaW4gdGhpcyBmaWxlLCBub3QgYSBjbGVhbiBydW4gKENPUkVfUlVMRVMgQXBwZW5kaXggQyAxMjogZXZlcnkKcnVsZSBuZWVkcyBhIGZsb29yIHRoYXQgIm5vdGhpbmciIGNhbm5vdCBzYXRpc2Z5KS4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAoKZnJvbSAuY29yZSBpbXBvcnQgcm5nX2ZvciwgdGVybWluYWxfc2VlZCwgU0lNX05PVywgQVVUSF9CQUNLRklMTF9EQVlTCgojIFJ1bGUgNSdzIGN1dG92ZXIgcG9pbnQgLS0gcmUtZXhwb3J0ZWQgaGVyZSBzbyBhbnkgZG93bnN0cmVhbSBjb25zdW1lciAoYSBLUUwKIyBmdW5jdGlvbiwgYSB2YWxpZGF0aW9uIHNjcmlwdCkgaGFzIG9uZSBwbGFjZSB0byByZWFkIGl0IGZyb20sIGV2ZW4gdGhvdWdoIHRoZQojIGN1dG92ZXIgaXRzZWxmIGlzIGFwcGxpZWQgaW5saW5lIGR1cmluZyBhdXRoX2V2ZW50cy5weSBnZW5lcmF0aW9uICh0aGUgZmllbGQKIyBoYXMgdG8gbm90IGV4aXN0IGJlZm9yZSB0aGUgY3V0b3Zlciwgd2hpY2ggaXMgYSBnZW5lcmF0aW9uLXRpbWUgZGVjaXNpb24sIG5vdAojIGEgcG9zdC1ob2MgbXV0YXRpb24pLgpBVVRIX1dJTkRPV19TVEFSVCA9IFNJTV9OT1cgLSBwZC5UaW1lZGVsdGEoZGF5cz1BVVRIX0JBQ0tGSUxMX0RBWVMpClNDSEVNQV9DVVRPVkVSID0gQVVUSF9XSU5ET1dfU1RBUlQgKyAwLjYgKiAoU0lNX05PVyAtIEFVVEhfV0lORE9XX1NUQVJUKQoKCmRlZiBkdXBsaWNhdGVfcm93cyhkZjogcGQuRGF0YUZyYW1lLCBybmc6IG5wLnJhbmRvbS5HZW5lcmF0b3IsIGZyYWM6IGZsb2F0ID0gMC4wMDQpIC0+IHR1cGxlW3BkLkRhdGFGcmFtZSwgaW50XToKICAgICIiIkRRIHJ1bGUgMTogaWRlbXBvdGVudCBkZWR1cGUgdGFyZ2V0LiBFeGFjdCBkdXBsaWNhdGVzIG9mIGEgcmFuZG9tIHNhbXBsZSwKICAgIGlkZW50aWNhbCBuYXR1cmFsIGtleSAoZS5nLiBhdXRoX2lkIC8gdGVsZW1ldHJ5X2lkKSBhbmQgZXZlcnkgb3RoZXIgZmllbGQuIiIiCiAgICBuID0gbWF4KDEsIGludChyb3VuZChsZW4oZGYpICogZnJhYykpKQogICAgZHVwX2lkeCA9IHJuZy5jaG9pY2UobGVuKGRmKSwgc2l6ZT1uLCByZXBsYWNlPUZhbHNlKQogICAgZHVwZXMgPSBkZi5pbG9jW2R1cF9pZHhdLmNvcHkoKQogICAgb3V0ID0gcGQuY29uY2F0KFtkZiwgZHVwZXNdLCBpZ25vcmVfaW5kZXg9VHJ1ZSkKICAgIHJldHVybiBvdXQsIG4KCgpkZWYgcmVvcmRlcl9yZXZlcnNhbHMoYXV0aF9kZjogcGQuRGF0YUZyYW1lLCBybmc6IG5wLnJhbmRvbS5HZW5lcmF0b3IsIGZyYWM6IGZsb2F0ID0gMC4wMikgLT4gdHVwbGVbcGQuRGF0YUZyYW1lLCBpbnRdOgogICAgIiIiRFEgcnVsZSAyOiBvdXQtb2Ytb3JkZXIgcmVzb2x1dGlvbi4gRm9yIGEgc2FtcGxlIG9mIHJldmVyc2FsL3BhcnRpYWxfY2FwdHVyZQogICAgcm93cywgc2V0IGluZ2VzdF90aW1lIGVhcmxpZXIgdGhhbiB0aGUgb3JpZ2luYWwgYXV0aF9pZCByb3cncyBvd24gaW5nZXN0X3RpbWUKICAgIC0tIHRoZSByZXNvbHV0aW9uIGdlbnVpbmVseSBhcnJpdmVzIGZpcnN0LiIiIgogICAgZGYgPSBhdXRoX2RmLmNvcHkoKQogICAgcmVzb2x1dGlvbnMgPSBkZi5pbmRleFtkZlsiZXZlbnRfdHlwZSJdLmlzaW4oWyJyZXZlcnNhbCIsICJwYXJ0aWFsX2NhcHR1cmUiXSldLnRvX251bXB5KCkKICAgIGlmIGxlbihyZXNvbHV0aW9ucykgPT0gMDoKICAgICAgICByZXR1cm4gZGYsIDAKICAgIG4gPSBtYXgoMSwgaW50KHJvdW5kKGxlbihyZXNvbHV0aW9ucykgKiBmcmFjKSkpCiAgICBuID0gbWluKG4sIGxlbihyZXNvbHV0aW9ucykpCiAgICBjaG9zZW4gPSBybmcuY2hvaWNlKHJlc29sdXRpb25zLCBzaXplPW4sIHJlcGxhY2U9RmFsc2UpCgogICAgIyBkZWR1cGUgb24gYXV0aF9pZCBiZWZvcmUgaW5kZXhpbmcgLS0gcnVsZSAxIChkdXBsaWNhdGVfcm93cykgcnVucyBmaXJzdCBpbgogICAgIyBhcHBseV9hbGxfZHFfZmF1bHRzLCBzbyBhdXRoX2lkIGlzIG5vdCBndWFyYW50ZWVkIHVuaXF1ZSBoZXJlOyBkdXBsaWNhdGVzCiAgICAjIGNyZWF0ZWQgYnkgcnVsZSAxIGFyZSBleGFjdCBjb3BpZXMgYXQgdGhpcyBwb2ludCwgc28gYW55IG9uZSBvZiB0aGVtCiAgICAjIGNhcnJpZXMgdGhlIGNvcnJlY3QgaW5nZXN0X3RpbWUgdG8gbG9vayB1cC4KICAgIG9yaWdpbmFsX2luZ2VzdCA9IGRmLmRyb3BfZHVwbGljYXRlcyhzdWJzZXQ9ImF1dGhfaWQiKS5zZXRfaW5kZXgoImF1dGhfaWQiKVsiaW5nZXN0X3RpbWUiXQogICAgYXBwbGllZCA9IDAKICAgIHJlbGF0ZWRfaWRzID0gZGYubG9jW2Nob3NlbiwgInJlbGF0ZWRfYXV0aF9pZCJdLnRvX251bXB5KCkKICAgIGxvb2t1cCA9IG9yaWdpbmFsX2luZ2VzdC5yZWluZGV4KHJlbGF0ZWRfaWRzKQogICAgdmFsaWQgPSB+bG9va3VwLmlzbmEoKS50b19udW1weSgpCiAgICBjaG9zZW5fdmFsaWQgPSBjaG9zZW5bdmFsaWRdCiAgICBsb29rdXBfdmFsaWQgPSBsb29rdXAudG9fbnVtcHkoKVt2YWxpZF0KICAgIGlmIGxlbihjaG9zZW5fdmFsaWQpOgogICAgICAgIGxlYWRfbWludXRlcyA9IHJuZy51bmlmb3JtKDEsIDEwLCBzaXplPWxlbihjaG9zZW5fdmFsaWQpKQogICAgICAgIG5ld19pbmdlc3QgPSAocGQudG9fZGF0ZXRpbWUobG9va3VwX3ZhbGlkKSAtIHBkLnRvX3RpbWVkZWx0YShsZWFkX21pbnV0ZXMsIHVuaXQ9Im0i"
    "KSkuYXN0eXBlKAogICAgICAgICAgICBkZlsiaW5nZXN0X3RpbWUiXS5kdHlwZQogICAgICAgICkKICAgICAgICBkZi5sb2NbY2hvc2VuX3ZhbGlkLCAiaW5nZXN0X3RpbWUiXSA9IG5ld19pbmdlc3QKICAgICAgICBhcHBsaWVkID0gbGVuKGNob3Nlbl92YWxpZCkKICAgIHJldHVybiBkZiwgYXBwbGllZAoKCmRlZiBsYXRlX2J1cnN0KGRmOiBwZC5EYXRhRnJhbWUsIHJuZzogbnAucmFuZG9tLkdlbmVyYXRvciwgd2luZG93X2RheXM6IGludCwgYnVyc3RzX3Blcl9kYXk6IGludCA9IDE1KSAtPiB0dXBsZVtwZC5EYXRhRnJhbWUsIGludF06CiAgICAiIiJEUSBydWxlIDM6IGxhdGUgYXJyaXZhbCBwYXN0IHRoZSB3YXRlcm1hcmsuIGBidXJzdHNfcGVyX2RheWAgKHRlcm1pbmFsLCBkYXkpCiAgICBwYWlycyBnZXQgYSA0MC1taW51dGUgY2x1c3RlciBvZiByb3dzIHdob3NlIGluZ2VzdF90aW1lIGlzIHB1c2hlZCAzNS01MAogICAgbWludXRlcyBhZnRlciBldmVudF90aW1lIC0tIG9uZSBidWZmZXJlZCBvZmZsaW5lIGR1bXAuCgogICAgR3JvdXBzIHJvdyBwb3NpdGlvbnMgYnkgdGVybWluYWxfaWQgT05DRSBzbyBlYWNoIG9mIHRoZSAoYnVyc3RzX3Blcl9kYXkgKgogICAgd2luZG93X2RheXMpIGl0ZXJhdGlvbnMgb25seSBzY2FucyB0aGF0IHRlcm1pbmFsJ3Mgb3duIHJvd3MgKH50aG91c2FuZHMpLAogICAgbm90IHRoZSBmdWxsIG11bHRpLW1pbGxpb24tcm93IGZyYW1lIC0tIHRoZSBuYWl2ZSBmdWxsLWFycmF5IG1hc2sgcGVyCiAgICBpdGVyYXRpb24gaXMgd2hhdCBtYWRlIHRoaXMgdGhlIHNsb3dlc3Qgc3RlcCBpbiB0aGUgcGlwZWxpbmUuIiIiCiAgICBkZiA9IGRmLmNvcHkoKQogICAgZXZlbnRfdGltZV9ucCA9IHBkLnRvX2RhdGV0aW1lKGRmWyJldmVudF90aW1lIl0pLnRvX251bXB5KCkKICAgIHRlcm1pbmFsX2dyb3VwcyA9IGRmLmdyb3VwYnkoInRlcm1pbmFsX2lkIiwgc29ydD1GYWxzZSwgb2JzZXJ2ZWQ9VHJ1ZSkuaW5kaWNlcwogICAgdGVybWluYWxzID0gbnAuYXJyYXkobGlzdCh0ZXJtaW5hbF9ncm91cHMua2V5cygpKSkKICAgIGlmIGxlbih0ZXJtaW5hbHMpID09IDA6CiAgICAgICAgcmV0dXJuIGRmLCAwCiAgICBuX2J1cnN0cyA9IGJ1cnN0c19wZXJfZGF5ICogd2luZG93X2RheXMKICAgIGFwcGxpZWQgPSAwCiAgICB3aW5kb3dfc3RhcnQgPSBldmVudF90aW1lX25wLm1pbigpCiAgICBpbmdlc3RfY29sID0gZGYuY29sdW1ucy5nZXRfbG9jKCJpbmdlc3RfdGltZSIpCiAgICBmb3IgXyBpbiByYW5nZShuX2J1cnN0cyk6CiAgICAgICAgdGVybSA9IHJuZy5jaG9pY2UodGVybWluYWxzKQogICAgICAgIHRlcm1faWR4ID0gdGVybWluYWxfZ3JvdXBzW3Rlcm1dCiAgICAgICAgZGF5X29mZnNldCA9IHJuZy5pbnRlZ2VycygwLCB3aW5kb3dfZGF5cykKICAgICAgICBob3VyX29mZnNldCA9IHJuZy51bmlmb3JtKDAsIDI0KQogICAgICAgIGJ1cnN0X3N0YXJ0ID0gd2luZG93X3N0YXJ0ICsgbnAudGltZWRlbHRhNjQoaW50KGRheV9vZmZzZXQpLCAiRCIpICsgbnAudGltZWRlbHRhNjQoCiAgICAgICAgICAgIGludChob3VyX29mZnNldCAqIDM2MDApLCAicyIKICAgICAgICApCiAgICAgICAgYnVyc3RfZW5kID0gYnVyc3Rfc3RhcnQgKyBucC50aW1lZGVsdGE2NCg0MCwgIm0iKQogICAgICAgIHRlcm1fdGltZXMgPSBldmVudF90aW1lX25wW3Rlcm1faWR4XQogICAgICAgIG1hc2sgPSAodGVybV90aW1lcyA+PSBidXJzdF9zdGFydCkgJiAodGVybV90aW1lcyA8PSBidXJzdF9lbmQpCiAgICAgICAgaWR4ID0gdGVybV9pZHhbbWFza10KICAgICAgICBpZiBsZW4oaWR4KSA9PSAwOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGRlbGF5X21pbiA9IHJuZy51bmlmb3JtKDM1LCA1MCwgc2l6ZT1sZW4oaWR4KSkKICAgICAgICBuZXdfaW5nZXN0ID0gKHBkLnRvX2RhdGV0aW1lKGV2ZW50X3RpbWVfbnBbaWR4XSkgKyBwZC50b190aW1lZGVsdGEoZGVsYXlfbWluLCB1bml0PSJtIikpLmFzdHlwZSgKICAgICAgICAgICAgZGZbImluZ2VzdF90aW1lIl0uZHR5cGUKICAgICAgICApCiAgICAgICAgZGYuaWxvY1tpZHgsIGluZ2VzdF9jb2xdID0gbmV3X2luZ2VzdAogICAgICAgIGFwcGxpZWQgKz0gbGVuKGlkeCkKICAgIHJldHVybiBkZiwgYXBwbGllZAoKCmRlZiBjbG9ja19za2V3KGRmOiBwZC5EYXRhRnJhbWUsIG1hc3Rlcl9zZWVkOiBpbnQgfCBOb25lID0gTm9uZSkgLT4gdHVwbGVbcGQuRGF0YUZyYW1lLCBpbnRdOgogICAgIiIiRFEgcnVsZSA0OiBkZXZpY2UgY2xvY2sgc2tldy4gRWFjaCB0ZXJtaW5hbCBkcmF3cyBvbmUgZml4ZWQgb2Zmc2V0IGluCiAgICBbLTUsICs1XSBtaW51dGVzLCBzZWVkZWQgcGVyIHRlcm1pbmFsIChzdGFibGUgYWNyb3NzIGl0cyB3aG9sZSBoaXN0b3J5IGFuZAogICAgaWRlbnRpY2FsIHdoaWNoZXZlciByb3V0ZSAtLSBiYWNrZmlsbCBvciBsaXZlLXJlcGxheSAtLSBnZW5lcmF0ZXMgaXQpLCBhbmQKICAgIHRoYXQgb2Zmc2V0IGlzIGFwcGxpZWQgdG8gZXZlbnRfdGltZS4gaW5nZXN0X3RpbWUgaXMgbGVmdCBhcyB0aGUKICAgICh1bnNrZXdlZCkgdHJ1ZSBhcnJpdmFsIHRpbWUsIHNvIHRoZSB0d28gY29sdW1ucyBub3cgZ2VudWluZWx5IGRpdmVyZ2UuIiIiCiAgICBkZiA9IGRmLmNvcHkoKQogICAgdGVybWluYWxzID0gZGZbInRlcm1pbmFsX2lkIl0udW5pcXVlKCkKICAgIG9mZnNldHMgPSB7fQogICAgZm9yIHQgaW4gdGVybWluYWxzOgogICAgICAgIHNlZWQgPSB0ZXJtaW5hbF9zZWVkKHN0cih0KSwgbWFzdGVyX3NlZWQpIGlmIG1hc3Rlcl9zZWVkIGlzIG5vdCBOb25lIGVsc2UgdGVybWluYWxfc2VlZChzdHIodCkpCiAgICAgICAgbG9jYWxfcm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKHNlZWQpCiAgICAgICAgb2Zmc2V0c1t0XSA9IGZsb2F0KGxvY2FsX3JuZy51bmlmb3JtKC01LCA1KSkKICAgIG9mZnNldF9taW51dGVzID0gZGZbInRlcm1pbmFsX2lkIl0ubWFwKG9mZnNldHMpLnRvX251bXB5KCkKICAgIG5ld19l"
    "dmVudF90aW1lID0gKHBkLnRvX2RhdGV0aW1lKGRmWyJldmVudF90aW1lIl0pICsgcGQudG9fdGltZWRlbHRhKG9mZnNldF9taW51dGVzLCB1bml0PSJtIikpLmFzdHlwZSgKICAgICAgICBkZlsiZXZlbnRfdGltZSJdLmR0eXBlCiAgICApCiAgICBkZlsiZXZlbnRfdGltZSJdID0gbmV3X2V2ZW50X3RpbWUKICAgIGFwcGxpZWQgPSBpbnQoKG9mZnNldF9taW51dGVzICE9IDApLnN1bSgpKQogICAgcmV0dXJuIGRmLCBhcHBsaWVkCgoKZGVmIGNvbGRfcGF0aF9nYXAoZGY6IHBkLkRhdGFGcmFtZSwgcm5nOiBucC5yYW5kb20uR2VuZXJhdG9yLCBrZXlfY29sOiBzdHIsIG46IGludCA9IDUwKToKICAgICIiIkRRIHJ1bGUgNjogaG90LXZzLWNvbGQgcmVjb25jaWxpYXRpb24uIFRoZSBjb2xkLXBhdGggKExha2Vob3VzZSkgY29weQogICAgZGVsaWJlcmF0ZWx5IGV4Y2x1ZGVzIGBuYCByb3dzIHRoYXQgdGhlIGhvdC1wYXRoIChFdmVudGhvdXNlKSBleHBvcnQKICAgIGluY2x1ZGVzLCBzbyB0aGUgcmVjb25jaWxpYXRpb24gY2hlY2sgKHJvdyBjb3VudCBhbmQgc3VtbWVkIGFtb3VudCwgcGVyCiAgICBzdHJlYW0pIGhhcyBhIGdlbnVpbmUsIGtub3duLCBub24temVybyBkaXNjcmVwYW5jeSB0byBmaW5kLiBSZXR1cm5zCiAgICAoaG90X2RmLCBjb2xkX2RmLCBleGNsdWRlZF9rZXlzX2RmKS4iIiIKICAgIG4gPSBtaW4obiwgbGVuKGRmKSkKICAgIGV4Y2xfaWR4ID0gcm5nLmNob2ljZShsZW4oZGYpLCBzaXplPW4sIHJlcGxhY2U9RmFsc2UpCiAgICBleGNsdWRlZF9rZXlzID0gZGYuaWxvY1tleGNsX2lkeF1bW2tleV9jb2xdXS5jb3B5KCkKICAgIGNvbGRfZGYgPSBkZi5kcm9wKGRmLmluZGV4W2V4Y2xfaWR4XSkucmVzZXRfaW5kZXgoZHJvcD1UcnVlKQogICAgcmV0dXJuIGRmLCBjb2xkX2RmLCBleGNsdWRlZF9rZXlzCgoKZGVmIGFwcGx5X2FsbF9kcV9mYXVsdHMoYXV0aF9ldmVudHM6IHBkLkRhdGFGcmFtZSwgdGVsZW1ldHJ5OiBwZC5EYXRhRnJhbWUsIG1hc3Rlcl9zZWVkOiBpbnQgfCBOb25lID0gTm9uZSk6CiAgICAiIiJPcmNoZXN0cmF0ZXMgcnVsZXMgMS00IGFuZCA2IGFnYWluc3QgYm90aCByYXcgc3RyZWFtcyAocnVsZSA1IGlzIGFwcGxpZWQKICAgIGF0IGdlbmVyYXRpb24gdGltZSBpbiBhdXRoX2V2ZW50cy5weSAtLSBzZWUgU0NIRU1BX0NVVE9WRVIgYWJvdmUpLiBSZXR1cm5zCiAgICAoYXV0aF9ob3QsIGF1dGhfY29sZCwgdGVsZW1ldHJ5X2hvdCwgdGVsZW1ldHJ5X2NvbGQsIGZhdWx0X2xvZywKICAgIGNvbGRfZXhjbHVzaW9ucykgd2hlcmUgZmF1bHRfbG9nIGlzIHRoZSBwZXItcnVsZSBjb3VudCByZWdpc3RlciB0aGF0IGxhbmRzCiAgICBpbiBMYW5kaW5nX01hbmlmZXN0LmNzdi4iIiIKICAgIHJuZyA9IHJuZ19mb3IoImRxX2ZhdWx0cyIsIG1hc3Rlcl9zZWVkKSBpZiBtYXN0ZXJfc2VlZCBpcyBub3QgTm9uZSBlbHNlIHJuZ19mb3IoImRxX2ZhdWx0cyIpCgogICAgYXV0aF9ldmVudHMsIG5fZHVwX2F1dGggPSBkdXBsaWNhdGVfcm93cyhhdXRoX2V2ZW50cywgcm5nLCBmcmFjPTAuMDA0KQogICAgYXV0aF9ldmVudHMsIG5fcmVvcmRlciA9IHJlb3JkZXJfcmV2ZXJzYWxzKGF1dGhfZXZlbnRzLCBybmcsIGZyYWM9MC4wMikKICAgIGF1dGhfZXZlbnRzLCBuX2xhdGVfYXV0aCA9IGxhdGVfYnVyc3QoYXV0aF9ldmVudHMsIHJuZywgd2luZG93X2RheXM9QVVUSF9CQUNLRklMTF9EQVlTLCBidXJzdHNfcGVyX2RheT0xNSkKICAgIGF1dGhfZXZlbnRzLCBuX3NrZXdfYXV0aCA9IGNsb2NrX3NrZXcoYXV0aF9ldmVudHMsIG1hc3Rlcl9zZWVkKQoKICAgIHRlbGVtZXRyeSwgbl9kdXBfdGVsID0gZHVwbGljYXRlX3Jvd3ModGVsZW1ldHJ5LCBybmcsIGZyYWM9MC4wMDQpCiAgICB0ZWxlbWV0cnksIG5fbGF0ZV90ZWwgPSBsYXRlX2J1cnN0KHRlbGVtZXRyeSwgcm5nLCB3aW5kb3dfZGF5cz0zMCwgYnVyc3RzX3Blcl9kYXk9MTUpCiAgICB0ZWxlbWV0cnksIG5fc2tld190ZWwgPSBjbG9ja19za2V3KHRlbGVtZXRyeSwgbWFzdGVyX3NlZWQpCgogICAgYXV0aF9ob3QsIGF1dGhfY29sZCwgYXV0aF9leGNsID0gY29sZF9wYXRoX2dhcChhdXRoX2V2ZW50cywgcm5nLCAiYXV0aF9pZCIsIG49NTApCiAgICB0ZWxfaG90LCB0ZWxfY29sZCwgdGVsX2V4Y2wgPSBjb2xkX3BhdGhfZ2FwKHRlbGVtZXRyeSwgcm5nLCAidGVsZW1ldHJ5X2lkIiwgbj01MCkKCiAgICBmYXVsdF9sb2cgPSB7CiAgICAgICAgInJ1bGUxX2R1cGxpY2F0ZV9hdXRoIjogbl9kdXBfYXV0aCwKICAgICAgICAicnVsZTFfZHVwbGljYXRlX3RlbGVtZXRyeSI6IG5fZHVwX3RlbCwKICAgICAgICAicnVsZTJfcmVvcmRlcl9yZXZlcnNhbHMiOiBuX3Jlb3JkZXIsCiAgICAgICAgInJ1bGUzX2xhdGVfYnVyc3RfYXV0aCI6IG5fbGF0ZV9hdXRoLAogICAgICAgICJydWxlM19sYXRlX2J1cnN0X3RlbGVtZXRyeSI6IG5fbGF0ZV90ZWwsCiAgICAgICAgInJ1bGU0X2Nsb2NrX3NrZXdfYXV0aF90ZXJtaW5hbHNfYWZmZWN0ZWQiOiBuX3NrZXdfYXV0aCwKICAgICAgICAicnVsZTRfY2xvY2tfc2tld190ZWxlbWV0cnlfdGVybWluYWxzX2FmZmVjdGVkIjogbl9za2V3X3RlbCwKICAgICAgICAicnVsZTVfc2NoZW1hX2N1dG92ZXJfYXBwbGllZF9hdCI6IHN0cihTQ0hFTUFfQ1VUT1ZFUiksCiAgICAgICAgInJ1bGU2X2NvbGRfcGF0aF9nYXBfYXV0aCI6IGxlbihhdXRoX2V4Y2wpLAogICAgICAgICJydWxlNl9jb2xkX3BhdGhfZ2FwX3RlbGVtZXRyeSI6IGxlbih0ZWxfZXhjbCksCiAgICB9CiAgICBjb2xkX2V4Y2x1c2lvbnMgPSB7ImF1dGhfZXZlbnRzIjogYXV0aF9leGNsLCAidGVybWluYWxfdGVsZW1ldHJ5IjogdGVsX2V4Y2x9CiAgICByZXR1cm4gYXV0aF9ob3QsIGF1dGhfY29sZCwgdGVsX2hvdCwgdGVsX2NvbGQsIGZhdWx0X2xvZywgY29sZF9leGNsdXNpb25zCg=="
)
_FILES["dq_faults.py"] = _b64_dq_faults_py
_b64_manifest_py = (
    "IiIiTGFuZGluZ19NYW5pZmVzdC5jc3YuIFNlZSBHRU5FUkFUT1JfU1BFQy5tZCBzZWN0aW9uIDcuCgpydW5faWQgaXMgYSBkZXRlcm1pbmlzdGljIGxhYmVsLCBuZXZlciBhIHdhbGwtY2xvY2sgc3RhbXAgLS0gdGhhdCBpcyB3aGF0IGtlZXBzCnRoaXMgZmlsZSBieXRlLWlkZW50aWNhbCBhY3Jvc3MgcmVwcm9kdWNpYmlsaXR5IHJ1bnMuIFJlYWwgd2FsbC1jbG9jayBwcm92ZW5hbmNlCmJlbG9uZ3MgaW4gX3J1bl9tZXRhLmpzb24sIHdyaXR0ZW4gYnkgdGhlIGNhbGxlciBvdXRzaWRlIHRoaXMgdHJlZS4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQganNvbgoKaW1wb3J0IHBhbmRhcyBhcyBwZAoKZnJvbSAuY29yZSBpbXBvcnQgR0VORVJBVE9SX1ZFUlNJT04sIE1BU1RFUl9TRUVECgpUSU1FX0NPTFVNTiA9IHsKICAgICJhdXRoX2V2ZW50cyI6ICJldmVudF90aW1lIiwKICAgICJ0ZXJtaW5hbF90ZWxlbWV0cnkiOiAiZXZlbnRfdGltZSIsCiAgICAiZGlzcHV0ZV9ldmVudHMiOiAiZGlzcHV0ZV90aW1lIiwKICAgICJncm91bmRfdHJ1dGgiOiAid2luZG93X3N0YXJ0IiwKICAgICJkaW1fbWVyY2hhbnQiOiBOb25lLAogICAgImRpbV9zdG9yZSI6IE5vbmUsCiAgICAiZGltX3Rlcm1pbmFsIjogTm9uZSwKICAgICJkaW1faXNzdWVyIjogTm9uZSwKICAgICJkaW1fc3RvcmVfY2FsZW5kYXIiOiBOb25lLAp9CgoKZGVmIGJ1aWxkX2xhbmRpbmdfbWFuaWZlc3QoCiAgICBzdHJlYW1zOiBkaWN0W3N0ciwgcGQuRGF0YUZyYW1lXSwKICAgIHJ1bl9pZDogc3RyLAogICAgZmF1bHRfbG9nOiBkaWN0IHwgTm9uZSA9IE5vbmUsCiAgICBtYXN0ZXJfc2VlZDogaW50IHwgTm9uZSA9IE5vbmUsCikgLT4gcGQuRGF0YUZyYW1lOgogICAgcm93cyA9IFtdCiAgICBlcGlzb2RlX2NvdW50cyA9IE5vbmUKICAgIGlmICJncm91bmRfdHJ1dGgiIGluIHN0cmVhbXMgYW5kIGxlbihzdHJlYW1zWyJncm91bmRfdHJ1dGgiXSk6CiAgICAgICAgZXBpc29kZV9jb3VudHMgPSBzdHJlYW1zWyJncm91bmRfdHJ1dGgiXVsiZXBpc29kZV90eXBlIl0udmFsdWVfY291bnRzKCkudG9fZGljdCgpCgogICAgZm9yIG5hbWUsIGRmIGluIHN0cmVhbXMuaXRlbXMoKToKICAgICAgICB0aW1lX2NvbCA9IFRJTUVfQ09MVU1OLmdldChuYW1lKQogICAgICAgIGlmIHRpbWVfY29sIGFuZCBsZW4oZGYpOgogICAgICAgICAgICBtaW5fdCA9IHN0cihkZlt0aW1lX2NvbF0ubWluKCkpCiAgICAgICAgICAgIG1heF90ID0gc3RyKGRmW3RpbWVfY29sXS5tYXgoKSkKICAgICAgICBlbHNlOgogICAgICAgICAgICBtaW5fdCA9ICIiCiAgICAgICAgICAgIG1heF90ID0gIiIKICAgICAgICByb3dzLmFwcGVuZCgKICAgICAgICAgICAgewogICAgICAgICAgICAgICAgInN0cmVhbV9uYW1lIjogbmFtZSwKICAgICAgICAgICAgICAgICJyb3dfY291bnQiOiBsZW4oZGYpLAogICAgICAgICAgICAgICAgIm1pbl9ldmVudF90aW1lIjogbWluX3QsCiAgICAgICAgICAgICAgICAibWF4X2V2ZW50X3RpbWUiOiBtYXhfdCwKICAgICAgICAgICAgICAgICJnZW5lcmF0b3JfdmVyc2lvbiI6IEdFTkVSQVRPUl9WRVJTSU9OLAogICAgICAgICAgICAgICAgIm1hc3Rlcl9zZWVkIjogbWFzdGVyX3NlZWQgaWYgbWFzdGVyX3NlZWQgaXMgbm90IE5vbmUgZWxzZSBNQVNURVJfU0VFRCwKICAgICAgICAgICAgICAgICJydW5faWQiOiBydW5faWQsCiAgICAgICAgICAgICAgICAiZmF1bHRfcnVsZXNfYXBwbGllZCI6IGpzb24uZHVtcHMoZmF1bHRfbG9nLCBzb3J0X2tleXM9VHJ1ZSkgaWYgZmF1bHRfbG9nIGFuZCBuYW1lIGluICgKICAgICAgICAgICAgICAgICAgICAiYXV0aF9ldmVudHMiLAogICAgICAgICAgICAgICAgICAgICJ0ZXJtaW5hbF90ZWxlbWV0cnkiLAogICAgICAgICAgICAgICAgKSBlbHNlICIiLAogICAgICAgICAgICAgICAgImVwaXNvZGVfY291bnRfYnlfdHlwZSI6IGpzb24uZHVtcHMoZXBpc29kZV9jb3VudHMsIHNvcnRfa2V5cz1UcnVlKQogICAgICAgICAgICAgICAgaWYgZXBpc29kZV9jb3VudHMgYW5kIG5hbWUgPT0gImdyb3VuZF90cnV0aCIKICAgICAgICAgICAgICAgIGVsc2UgIiIsCiAgICAgICAgICAgIH0KICAgICAgICApCiAgICBtYW5pZmVzdCA9IHBkLkRhdGFGcmFtZShyb3dzKS5zb3J0X3ZhbHVlcygic3RyZWFtX25hbWUiKS5yZXNldF9pbmRleChkcm9wPVRydWUpCiAgICByZXR1cm4gbWFuaWZlc3QK"
)
_FILES["manifest.py"] = _b64_manifest_py
_b64_schemas_py = (
    "IiIiUHlBcnJvdyBzY2hlbWFzIC0tIHNpbmdsZSBzb3VyY2Ugb2YgdHJ1dGggZm9yIGNvbHVtbiBvcmRlci90eXBlcyBhY3Jvc3MgdGhlCmdlbmVyYXRvciwgdGhlIG5vdGVib29rIGJhY2tmaWxsIHJvdXRlIGFuZCB0aGUgUGFycXVldCBmaWxlcyBhIEtRTCAuaW5nZXN0IG9yIGEKTGFrZWhvdXNlIHNob3J0Y3V0IHdpbGwgcmVhZC4gU2VlIEdFTkVSQVRPUl9TUEVDLm1kIHNlY3Rpb24gNC4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgcHlhcnJvdyBhcyBwYQoKQVVUSF9FVkVOVFNfU0NIRU1BID0gcGEuc2NoZW1hKAogICAgWwogICAgICAgICgiYXV0aF9pZCIsIHBhLnN0cmluZygpKSwKICAgICAgICAoImV2ZW50X3R5cGUiLCBwYS5zdHJpbmcoKSksCiAgICAgICAgKCJyZWxhdGVkX2F1dGhfaWQiLCBwYS5zdHJpbmcoKSksCiAgICAgICAgKCJldmVudF90aW1lIiwgcGEudGltZXN0YW1wKCJ1cyIpKSwKICAgICAgICAoImluZ2VzdF90aW1lIiwgcGEudGltZXN0YW1wKCJ1cyIpKSwKICAgICAgICAoInRlcm1pbmFsX2lkIiwgcGEuc3RyaW5nKCkpLAogICAgICAgICgic3RvcmVfaWQiLCBwYS5zdHJpbmcoKSksCiAgICAgICAgKCJtZXJjaGFudF9pZCIsIHBhLnN0cmluZygpKSwKICAgICAgICAoImlzc3Vlcl9iaW4iLCBwYS5zdHJpbmcoKSksCiAgICAgICAgKCJjYXJkX3Rva2VuIiwgcGEuc3RyaW5nKCkpLAogICAgICAgICgiYW1vdW50IiwgcGEuZmxvYXQ2NCgpKSwKICAgICAgICAoImN1cnJlbmN5IiwgcGEuc3RyaW5nKCkpLAogICAgICAgICgibWNjIiwgcGEuc3RyaW5nKCkpLAogICAgICAgICgiYXV0aF9yZXN1bHQiLCBwYS5zdHJpbmcoKSksCiAgICAgICAgKCJkZWNsaW5lX3JlYXNvbiIsIHBhLnN0cmluZygpKSwKICAgICAgICAoInBvc19lbnRyeV9tb2RlIiwgcGEuc3RyaW5nKCkpLAogICAgICAgICgiaXNfY2FyZF9wcmVzZW50IiwgcGEuYm9vbF8oKSksCiAgICAgICAgKCJzY2FfZmxhZyIsIHBhLmJvb2xfKCkpLAogICAgICAgICgic2NoZW1hX3ZlcnNpb24iLCBwYS5pbnQzMigpKSwKICAgIF0KKQoKVEVSTUlOQUxfVEVMRU1FVFJZX1NDSEVNQSA9IHBhLnNjaGVtYSgKICAgIFsKICAgICAgICAoInRlbGVtZXRyeV9pZCIsIHBhLnN0cmluZygpKSwKICAgICAgICAoImV2ZW50X3RpbWUiLCBwYS50aW1lc3RhbXAoInVzIikpLAogICAgICAgICgiaW5nZXN0X3RpbWUiLCBwYS50aW1lc3RhbXAoInVzIikpLAogICAgICAgICgidGVybWluYWxfaWQiLCBwYS5zdHJpbmcoKSksCiAgICAgICAgKCJzdG9yZV9pZCIsIHBhLnN0cmluZygpKSwKICAgICAgICAoIm1lcmNoYW50X2lkIiwgcGEuc3RyaW5nKCkpLAogICAgICAgICgiaGVhcnRiZWF0X29rIiwgcGEuYm9vbF8oKSksCiAgICAgICAgKCJ0YW1wZXJfZmxhZyIsIHBhLmJvb2xfKCkpLAogICAgICAgICgiYmF0dGVyeV9wY3QiLCBwYS5mbG9hdDY0KCkpLAogICAgICAgICgic2lnbmFsX3N0cmVuZ3RoIiwgcGEuZmxvYXQ2NCgpKSwKICAgICAgICAoInNjaGVtYV92ZXJzaW9uIiwgcGEuaW50MzIoKSksCiAgICBdCikKCkRJU1BVVEVfRVZFTlRTX1NDSEVNQSA9IHBhLnNjaGVtYSgKICAgIFsKICAgICAgICAoImRpc3B1dGVfaWQiLCBwYS5zdHJpbmcoKSksCiAgICAgICAgKCJhdXRoX2lkIiwgcGEuc3RyaW5nKCkpLAogICAgICAgICgiZGlzcHV0ZV90aW1lIiwgcGEudGltZXN0YW1wKCJ1cyIpKSwKICAgICAgICAoImRpc3B1dGVfcmVhc29uIiwgcGEuc3RyaW5nKCkpLAogICAgICAgICgiZGlzcHV0ZV9vdXRjb21lIiwgcGEuc3RyaW5nKCkpLAogICAgICAgICgiYW1vdW50IiwgcGEuZmxvYXQ2NCgpKSwKICAgIF0KKQoKR1JPVU5EX1RSVVRIX1NDSEVNQSA9IHBhLnNjaGVtYSgKICAgIFsKICAgICAgICAoImVwaXNvZGVfaWQiLCBwYS5zdHJpbmcoKSksCiAgICAgICAgKCJlcGlzb2RlX3R5cGUiLCBwYS5zdHJpbmcoKSksCiAgICAgICAgKCJhZmZlY3RlZF9lbnRpdHlfdHlwZSIsIHBhLnN0cmluZygpKSwKICAgICAgICAoImFmZmVjdGVkX2VudGl0eV9pZCIsIHBhLnN0cmluZygpKSwKICAgICAgICAoIndpbmRvd19zdGFydCIsIHBhLnRpbWVzdGFtcCgidXMiKSksCiAgICAgICAgKCJ3aW5kb3dfZW5kIiwgcGEudGltZXN0YW1wKCJ1cyIpKSwKICAgICAgICAoImludGVuc2l0eV9wYXJhbV8xIiwgcGEuZmxvYXQ2NCgpKSwKICAgICAgICAoImludGVuc2l0eV9wYXJhbV8yIiwgcGEuZmxvYXQ2NCgpKSwKICAgICAgICAoInJvd19jb3VudF9oaW50IiwgcGEuaW50NjQoKSksCiAgICBdCikKCkRJTV9NRVJDSEFOVF9TQ0hFTUEgPSBwYS5zY2hlbWEoCiAgICBbCiAgICAgICAgKCJtZXJjaGFudF9pZCIsIHBhLnN0cmluZygpKSwKICAgICAgICAoIm1lcmNoYW50X3RpZXIiLCBwYS5zdHJpbmcoKSksCiAgICAgICAgKCJtY2MiLCBwYS5zdHJpbmcoKSksCiAgICAgICAgKCJtY2NfZGVzY3JpcHRpb24iLCBwYS5zdHJpbmcoKSksCiAgICBdCikKCkRJTV9TVE9SRV9TQ0hFTUEgPSBwYS5zY2hlbWEoCiAgICBbCiAgICAgICAgKCJzdG9yZV9pZCIsIHBhLnN0cmluZygpKSwKICAgICAgICAoIm1lcmNoYW50X2lkIiwgcGEuc3RyaW5nKCkpLAogICAgXQopCgpESU1fVEVSTUlOQUxfU0NIRU1BID0gcGEuc2NoZW1hKAogICAgWwogICAgICAgICgidGVybWluYWxfaWQiLCBwYS5zdHJpbmcoKSksCiAgICAgICAgKCJzdG9yZV9pZCIsIHBhLnN0cmluZygpKSwKICAgICAgICAoIm1lcmNoYW50X2lkIiwgcGEuc3RyaW5nKCkpLAogICAgXQopCgpESU1fSVNTVUVSX1NDSEVNQSA9IHBhLnNjaGVtYSgKICAgIFsKICAgICAgICAoImlzc3Vlcl9pZCIsIHBhLnN0cmluZygpKSwKICAgICAgICAoImlzc3Vlcl9iaW4iLCBwYS5zdHJpbmcoKSksCiAgICAgICAgKCJpc3N1ZXJfbmFtZSIsIHBhLnN0cmluZygpKSwKICAgIF0KKQoKRElNX1NUT1JFX0NBTEVOREFSX1NDSEVNQSA9IHBhLnNjaGVtYSgKICAgIFsKICAgICAgICAoInN0b3JlX2lkIiwgcGEuc3RyaW5nKCkpLAogICAgICAgICgiZGF5X29mX3dlZWsiLCBw"
    "YS5pbnQzMigpKSwKICAgICAgICAoImRheV9uYW1lIiwgcGEuc3RyaW5nKCkpLAogICAgICAgICgic2Vzc2lvbl9ubyIsIHBhLmludDMyKCkpLAogICAgICAgICgiaXNfY2xvc2VkIiwgcGEuYm9vbF8oKSksCiAgICAgICAgKCJvcGVuX3RpbWUiLCBwYS5zdHJpbmcoKSksCiAgICAgICAgKCJjbG9zZV90aW1lIiwgcGEuc3RyaW5nKCkpLAogICAgXQopCgpTQ0hFTUFTID0gewogICAgImF1dGhfZXZlbnRzIjogQVVUSF9FVkVOVFNfU0NIRU1BLAogICAgInRlcm1pbmFsX3RlbGVtZXRyeSI6IFRFUk1JTkFMX1RFTEVNRVRSWV9TQ0hFTUEsCiAgICAiZGlzcHV0ZV9ldmVudHMiOiBESVNQVVRFX0VWRU5UU19TQ0hFTUEsCiAgICAiZ3JvdW5kX3RydXRoIjogR1JPVU5EX1RSVVRIX1NDSEVNQSwKICAgICJkaW1fbWVyY2hhbnQiOiBESU1fTUVSQ0hBTlRfU0NIRU1BLAogICAgImRpbV9zdG9yZSI6IERJTV9TVE9SRV9TQ0hFTUEsCiAgICAiZGltX3Rlcm1pbmFsIjogRElNX1RFUk1JTkFMX1NDSEVNQSwKICAgICJkaW1faXNzdWVyIjogRElNX0lTU1VFUl9TQ0hFTUEsCiAgICAiZGltX3N0b3JlX2NhbGVuZGFyIjogRElNX1NUT1JFX0NBTEVOREFSX1NDSEVNQSwKfQo="
)
_FILES["schemas.py"] = _b64_schemas_py
for _fname, _b64 in _FILES.items():
    with open(os.path.join(GEN_DIR, _fname), 'wb') as _f:
        _f.write(base64.b64decode(_b64))
print(f'Wrote {len(_FILES)} generator modules to {GEN_DIR}')

In [ ]:
# Cell 4 -- import the generator and run it fully in memory (no local
# parquet write -- this notebook ingests straight from the DataFrames).
import sys
sys.path.insert(0, "/tmp/p25_generator")

from generator.core import GENERATOR_VERSION
from generator.estate import build_dim_estate
from generator.trading_calendar import build_trading_hours
from generator.episodes import inject_episodes
from generator.auth_events import generate_auth_events
from generator.telemetry import generate_telemetry
from generator.disputes import generate_disputes
from generator.dq_faults import apply_all_dq_faults
from generator.manifest import build_landing_manifest

print(f"Generator version {GENERATOR_VERSION}, seed {MASTER_SEED}")

dim_merchant, dim_store, dim_terminal, dim_issuer = build_dim_estate(MASTER_SEED)
dim_store_calendar, archetype_map = build_trading_hours(dim_store, MASTER_SEED)
ground_truth = inject_episodes(dim_terminal, dim_issuer, archetype_map, MASTER_SEED)

auth_events = generate_auth_events(
    dim_merchant, dim_store, dim_terminal, dim_issuer, archetype_map, ground_truth, MASTER_SEED
)
telemetry = generate_telemetry(dim_terminal, ground_truth, MASTER_SEED)
dispute_events = generate_disputes(auth_events, ground_truth, MASTER_SEED)

auth_hot, auth_cold, tel_hot, tel_cold, fault_log, cold_exclusions = apply_all_dq_faults(
    auth_events, telemetry, MASTER_SEED
)

# Free the pre-fault originals immediately. apply_all_dq_faults has already
# copied what it needs into the hot/cold pairs, so holding auth_events and
# telemetry as well roughly doubles peak RSS for no benefit. This matters:
# generating the full set under pandas 2.1.4 was OOM-killed (exit 137) on a
# 7 GB box during Session 3 testing, where pandas 3.0 fitted. Your Fabric node
# is probably larger, but the headroom is worth having.
import gc
del auth_events, telemetry
gc.collect()

if ROW_LIMIT is not None:
    print(f"ROW_LIMIT={ROW_LIMIT} -- SMOKE TEST, not the full backfill")
    auth_hot = auth_hot.head(ROW_LIMIT)
    auth_cold = auth_cold.head(ROW_LIMIT)
    tel_hot = tel_hot.head(ROW_LIMIT)
    tel_cold = tel_cold.head(ROW_LIMIT)
    dispute_events = dispute_events.head(min(ROW_LIMIT, len(dispute_events)))

streams_for_manifest = {
    "auth_events": auth_hot, "terminal_telemetry": tel_hot,
    "dispute_events": dispute_events, "ground_truth": ground_truth,
    "dim_merchant": dim_merchant, "dim_store": dim_store,
    "dim_terminal": dim_terminal, "dim_issuer": dim_issuer,
    "dim_store_calendar": dim_store_calendar,
}
manifest = build_landing_manifest(streams_for_manifest, RUN_ID, fault_log, MASTER_SEED)
print(manifest.to_string())
print(f"Total rows this run: {sum(len(v) for v in streams_for_manifest.values())}")


In [ ]:
# Cell 5 -- Kusto connections using this notebook's own AAD context.
# notebookutils.credentials.getToken() is the standard Fabric-notebook pattern
# for authenticating to an Eventhouse/KQL database in the same tenant without a
# pasted connection string or service principal secret.
#
# TWO clients, because there are two endpoints:
#   kusto_client  -> CLUSTER_URI, for .create / .show / queries
#   ingest_client -> INGEST_URI,  for QueuedIngestClient
# One token serves both: the audience is the same Kusto resource.
from azure.kusto.data import KustoConnectionStringBuilder, KustoClient
from azure.kusto.data.data_format import DataFormat
from azure.kusto.ingest import QueuedIngestClient, IngestionProperties, ReportLevel

_ingest_uri = INGEST_URI.strip() if INGEST_URI else ""
if not _ingest_uri:
    _ingest_uri = CLUSTER_URI.replace("https://", "https://ingest-", 1)
    print(f"INGEST_URI blank -- derived as: {_ingest_uri}")

token = notebookutils.credentials.getToken(CLUSTER_URI)
kcsb_query = KustoConnectionStringBuilder.with_aad_user_token_authentication(CLUSTER_URI, token)
kcsb_ingest = KustoConnectionStringBuilder.with_aad_user_token_authentication(_ingest_uri, token)

kusto_client = KustoClient(kcsb_query)
ingest_client = QueuedIngestClient(kcsb_ingest)

# Prove the query endpoint works BEFORE generating 9.7m rows against it. A
# failure here is a 2-second failure; the same failure discovered in Cell 7 is
# a 5-minute one. This also confirms the KQL layer is actually present -- if
# the table count is not 18, the DDL did not all run and ingestion would land
# in a half-built database.
_probe = kusto_client.execute_mgmt(DATABASE, ".show tables | count")
_table_count = _probe.primary_results[0][0]["Count"]
print(f"Connected. {DATABASE} reports {_table_count} tables (expect 18).")
if _table_count != 18:
    print("  WARNING: expected 18 tables. Re-check 02_KQL files 00, 01, 03 before ingesting.")
print(f"Query  endpoint: {CLUSTER_URI}")
print(f"Ingest endpoint: {_ingest_uri}")


In [ ]:
# Cell 6 -- ingestion helper. Batches large DataFrames, reports progress,
# and NEVER silently drops a column mismatch -- it asserts the DataFrame's
# columns exactly match the target table's expected order before ingesting,
# because Kusto's default CSV-mapped ingestion is positional.
import time

def ingest_dataframe(df, table_name, expected_columns, batch_rows=BATCH_ROWS):
    missing = set(expected_columns) - set(df.columns)
    extra = set(df.columns) - set(expected_columns)
    if missing:
        raise ValueError(f"{table_name}: DataFrame is missing columns {missing}")
    if extra:
        print(f"WARNING {table_name}: dropping extra columns not in target schema: {extra}")
    df = df[expected_columns]
    props = IngestionProperties(
        database=DATABASE, table=table_name,
        data_format=DataFormat.CSV, report_level=ReportLevel.FailuresAndSuccesses,
    )
    n = len(df)
    if n == 0:
        print(f"{table_name}: 0 rows, nothing to ingest")
        return
    t0 = time.time()
    for start in range(0, n, batch_rows):
        chunk = df.iloc[start:start + batch_rows]
        ingest_client.ingest_from_dataframe(chunk, ingestion_properties=props)
        print(f"{table_name}: queued rows {start:,}-{start+len(chunk):,} of {n:,}")
    print(f"{table_name}: {n:,} rows queued in {time.time()-t0:.1f}s "
          f"(QUEUED, not yet visible -- ingestion is async; wait 1-3 min then "
          f"'{table_name} | count' in the KQL Queryset before trusting a count)")


In [ ]:
# Cell 7 -- run the ingestion, one call per table.
#
# COLUMN LISTS ARE TAKEN FROM generator/schemas.py, NOT from the spec's prose.
# They are asserted against it below rather than trusted: an earlier revision of
# this notebook carried hand-copied lists that disagreed with the generator in
# five places (ground_truth's intensity columns, dim_store's non-existent
# store_archetype, dim_issuer's issuer_id, dim_merchant's order and
# mcc_description, and the whole shape of dim_store_calendar). Kusto CSV
# ingestion is POSITIONAL, so a wrong order does not error -- it silently
# writes values into the wrong columns. Hence the assert.
from generator.schemas import SCHEMAS

AUTH_COLS   = [f.name for f in SCHEMAS["auth_events"]]
TEL_COLS    = [f.name for f in SCHEMAS["terminal_telemetry"]]
DSP_COLS    = [f.name for f in SCHEMAS["dispute_events"]]
GT_COLS     = [f.name for f in SCHEMAS["ground_truth"]]
MER_COLS    = [f.name for f in SCHEMAS["dim_merchant"]]
STORE_COLS  = [f.name for f in SCHEMAS["dim_store"]]
TERM_COLS   = [f.name for f in SCHEMAS["dim_terminal"]]
ISSUER_COLS = [f.name for f in SCHEMAS["dim_issuer"]]
CAL_COLS    = [f.name for f in SCHEMAS["dim_store_calendar"]]

# These must match 02_KQL/00_raw_and_staging_tables.kql Rev 2 exactly, in order.
EXPECTED = {
    "auth_events": ["auth_id","event_type","related_auth_id","event_time","ingest_time",
        "terminal_id","store_id","merchant_id","issuer_bin","card_token","amount",
        "currency","mcc","auth_result","decline_reason","pos_entry_mode",
        "is_card_present","sca_flag","schema_version"],
    "terminal_telemetry": ["telemetry_id","event_time","ingest_time","terminal_id",
        "store_id","merchant_id","heartbeat_ok","tamper_flag","battery_pct",
        "signal_strength","schema_version"],
    "dispute_events": ["dispute_id","auth_id","dispute_time","dispute_reason",
        "dispute_outcome","amount"],
    "ground_truth": ["episode_id","episode_type","affected_entity_type",
        "affected_entity_id","window_start","window_end","intensity_param_1",
        "intensity_param_2","row_count_hint"],
    "dim_merchant": ["merchant_id","merchant_tier","mcc","mcc_description"],
    "dim_store": ["store_id","merchant_id"],
    "dim_terminal": ["terminal_id","store_id","merchant_id"],
    "dim_issuer": ["issuer_id","issuer_bin","issuer_name"],
    "dim_store_calendar": ["store_id","day_of_week","day_name","session_no",
        "is_closed","open_time","close_time"],
}
for _name, _expected in EXPECTED.items():
    _actual = [f.name for f in SCHEMAS[_name]]
    assert _actual == _expected, (
        f"SCHEMA DRIFT on {_name}:\n  generator: {_actual}\n  KQL DDL  : {_expected}\n"
        f"Fix 02_KQL/00_raw_and_staging_tables.kql to match the generator, then re-run.")
print("Schema check passed -- all 9 streams match the KQL DDL, in order.")

# The four large frames are ingested then freed one at a time. Row counts are
# captured BEFORE deleting, because the reconciliation in the next cell needs
# them and the frame will be gone.
import gc
_ingested = {}

ingest_dataframe(auth_hot, "raw_auth_events", AUTH_COLS)
_ingested["raw_auth_events"] = len(auth_hot)
del auth_hot; gc.collect()

ingest_dataframe(auth_cold, "raw_auth_events_cold", AUTH_COLS)
_ingested["raw_auth_events_cold"] = len(auth_cold)
del auth_cold; gc.collect()

ingest_dataframe(tel_hot, "raw_terminal_telemetry", TEL_COLS)
_ingested["raw_terminal_telemetry"] = len(tel_hot)
del tel_hot; gc.collect()

ingest_dataframe(tel_cold, "raw_terminal_telemetry_cold", TEL_COLS)
_ingested["raw_terminal_telemetry_cold"] = len(tel_cold)
del tel_cold; gc.collect()

ingest_dataframe(dispute_events, "raw_dispute_events", DSP_COLS)
ingest_dataframe(ground_truth, "ground_truth", GT_COLS)
ingest_dataframe(dim_merchant, "stg_dim_merchant", MER_COLS)
ingest_dataframe(dim_store, "stg_dim_store", STORE_COLS)
ingest_dataframe(dim_terminal, "stg_dim_terminal", TERM_COLS)
ingest_dataframe(dim_issuer, "stg_dim_issuer", ISSUER_COLS)
ingest_dataframe(dim_store_calendar, "dim_store_calendar", CAL_COLS)

print()
print("All ingestion calls queued. Queued ingestion is ASYNC -- wait 1-3 minutes,")
print("then reconcile row counts in the KQL Queryset against the manifest Cell 4")
print("printed (SESSION_03_PORTAL_STEPS.md section C has the query block).")
print("If a table is still 0 after 5+ minutes, check the failure queue:")
print("  .show ingestion failures | where Database == 'EH_MeridianPay' | top 20 by FailedOn desc")


In [ ]:
# Cell 8 -- durable provenance: write Landing_Manifest.csv into a KQL table
# too, since no Lakehouse exists yet in this project as of Session 3 (see the
# Session 3 handover note -- component 8, "Lakehouse (cold path only)", has
# not been created; this notebook's cold-path handling above lands into
# raw_*_cold KQL tables as a Session 3 shortcut, not the Lakehouse the
# contract's component list describes. Flagged, not silently substituted --
# reconcile this when component 8 is actually built).
create_cmd = """.create-merge table landing_manifest (
    stream_name: string, row_count: string, min_event_time: string, max_event_time: string,
    generator_version: string, master_seed: string, run_id: string,
    fault_rules_applied: string, episode_count_by_type: string
)"""
kusto_client.execute_mgmt(DATABASE, create_cmd)

manifest_ingest = manifest.copy()
# every column as string -- the manifest's fault_rules_applied /
# episode_count_by_type cells are JSON blobs, and row_count is the only numeric
# one worth typing. Keeping them all string avoids a positional CSV type
# mismatch on a 9-row provenance table where types buy nothing.
manifest_ingest = manifest_ingest.astype(str)
manifest_props = IngestionProperties(database=DATABASE, table="landing_manifest", data_format=DataFormat.CSV)
ingest_client.ingest_from_dataframe(manifest_ingest, ingestion_properties=manifest_props)
print("landing_manifest row set queued. Verify with: landing_manifest | where run_id == '" + RUN_ID + "'")
print()
print("ALSO save a copy of the manifest print-out from Cell 4's output to")
print("01_Source/Landing_Manifest_session3.csv in the repo (Git integration")
print("captures the notebook DEFINITION automatically, not its run OUTPUT --")
print("copy the printed table by hand, or download this notebook's /tmp file")
print("if you re-enable local parquet writes).")
